In [1]:

import pandas as pd
import json

# مسیر فایل
file_path = r"D:\progam\Laqaee\Oman-Laws_RAG_Phase_1.json"

# 1. خوندن فایل JSON
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 2. تبدیل به DataFrame
if isinstance(data, list):
    df = pd.DataFrame(data)
elif isinstance(data, dict):
    first_key = list(data.keys())[0]
    df = pd.DataFrame(data[first_key])



text_dupes = df[df.duplicated(subset=["text"], keep=False)]
text_counts = text_dupes.groupby("text").size().reset_index(name="count").sort_values(by="count", ascending=False)



# 5. تعداد کل تکراری‌ها
print("تعداد رکوردها:", len(df))
print("تعداد رکوردهای تکراری بر اساس ستون متن:", len(text_dupes))

# حذف رکوردهای تکراری بر اساس ستون 'text' و نگه داشتن اولین رکورد
df_unique = df.drop_duplicates(subset=["text"], keep="first")

# تعداد رکوردها بعد از حذف تکراری‌ها
print("تعداد رکوردها بعد از حذف تکراری‌ها:", len(df_unique))

# حذف رکوردهایی که ستون 'text' خالی یا NaN هست
df_clean = df_unique.dropna(subset=['text'])

# همچنین می‌تونیم رکوردهایی که طول متن صفره هم حذف کنیم
df_clean = df_clean[df_clean['text'].str.strip() != ""]

print("تعداد رکوردها بعد از حذف متن‌های خالی:", len(df_clean))


تعداد رکوردها: 12547
تعداد رکوردهای تکراری بر اساس ستون متن: 2897
تعداد رکوردها بعد از حذف تکراری‌ها: 10537
تعداد رکوردها بعد از حذف متن‌های خالی: 10535


In [2]:
print("ستون‌ها:")
print(df_clean.columns.tolist())

print("\nتعداد رکوردها:", len(df_clean))

print("\nنوع داده هر ستون:")
print(df_clean.dtypes)

print("\nتعداد مقادیر خالی در هر ستون:")
print(df_clean.isna().sum())


ستون‌ها:
['json_link', 'short_link', 'canonical_link', 'date', 'title', 'text', 'html', 'error']

تعداد رکوردها: 10535

نوع داده هر ستون:
json_link         object
short_link        object
canonical_link    object
date              object
title             object
text              object
html              object
error             object
dtype: object

تعداد مقادیر خالی در هر ستون:
json_link             0
short_link            0
canonical_link        0
date                  0
title                 0
text                  0
html                  0
error             10535
dtype: int64


In [3]:
import re
import pandas as pd


def clean_text(text):
    # حذف فاصله اضافی در ابتدا و انتهای متن
    text = text.strip()
    # جایگزینی چند فاصله متوالی با یک فاصله
    text = re.sub(r'\s+', ' ', text)
    # حذف خطوط خالی یا فقط شامل فاصله
    text = re.sub(r'\n\s*\n', '\n', text)

    return text

def clean_header(text):
    text = re.sub(r'^(?:تحميل|English|نص معدل|\s)+', '', text)
    return text


noise_pattern = re.compile(
    r"""
    [\s\.]*                                   # فاصله یا نقطه قبل نویز
    (                                        # بلاک نویز
        (?:
            [0-9\u0660-\u0669]{1,4}           # عدد ۱ تا ۴ رقمی
            \s*/\s*
            [0-9\u0660-\u0669]{1,4}           # عدد ۱ تا ۴ رقمی
            \s*
        ){2,}                                 # حداقل دو تکرار
    )
    $                                         # فقط انتهای متن
    """,
    re.VERBOSE
)

def remove_trailing_numeric_noise(text):
    if not isinstance(text, str):
        return text
    return re.sub(noise_pattern, "", text).strip()
from datetime import datetime

AR_MONTHS = {
    "يناير": 1,
    "فبراير": 2,
    "مارس": 3,
    "أبريل": 4,
    "ابريل": 4,
    "مايو": 5,
    "يونيو": 6,
    "يوليو": 7,
    "أغسطس": 8,
    "اغسطس": 8,
    "سبتمبر": 9,
    "أكتوبر": 10,
    "اكتوبر": 10,
    "نوفمبر": 11,
    "ديسمبر": 12,
}

def parse_arabic_date(text):
    if not isinstance(text, str):
        return pd.NaT

    # حذف عبارت ثابت
    text = text.replace("تاريخ المقالة", "").strip()

    # استخراج روز، ماه، سال
    m = re.search(r"(\d{1,2})\s+([^\s]+)\s+(\d{4})", text)
    if not m:
        return pd.NaT

    day = int(m.group(1))
    month_ar = m.group(2)
    year = int(m.group(3))

    month = AR_MONTHS.get(month_ar)
    if not month:
        return pd.NaT

    try:
        return datetime(year, month, day)
    except ValueError:
        return pd.NaT

# اعمال تابع تمیزسازی روی ستون
df_clean['text'] = df_clean['text'].apply(clean_text)
df_clean['text'] = df_clean['text'].apply(clean_header)
df_clean['text'] = df_clean['text'].apply(remove_trailing_numeric_noise)

df_clean['date'] = df_clean['date'].apply(clean_text)
df_clean["date"] = df_clean["date"].apply(parse_arabic_date)

df_clean['title'] = df_clean['title'].apply(clean_text)
df_clean['html'] = df_clean['html'].apply(clean_text)

In [4]:
# اضافه کردن ستون متن و تعداد کلمات
df_clean['text_word_count'] = df_clean['text'].apply(lambda x: len(x.split()))
# ابتدا تعداد کلمات هر عنوان را محاسبه کن
df_clean['title_word_count'] = df_clean['title'].apply(lambda x: len(str(x).split()))


print("\nآمار طول متن:")
print(df_clean['text_word_count'].describe())
print("\nآمار طول عنوان:")
print(df_clean['title_word_count'].describe())



آمار طول متن:
count     10535.000000
mean        689.552159
std        3050.563075
min           0.000000
25%          14.000000
50%         133.000000
75%         356.000000
max      220969.000000
Name: text_word_count, dtype: float64

آمار طول عنوان:
count    10535.000000
mean        16.159279
std          6.188719
min          2.000000
25%         12.000000
50%         16.000000
75%         20.000000
max         66.000000
Name: title_word_count, dtype: float64


In [5]:
short_texts = df_clean[(df_clean['text_word_count'] > 90) & (df_clean['text_word_count'] < 100)]

print("تعداد قوانین خیلی کوتاه:", len(short_texts))

for i, row in short_texts.head().iterrows():
    print("-------------------")
    print("Title:", row['title'])
    print("Word count:", row['text_word_count'])
    print("id:",row['canonical_link'])
    print("Text:")
    print(row['text'])


تعداد قوانین خیلی کوتاه: 207
-------------------
Title: مرسوم سلطاني رقم ٦ / ٧٤ بخصوص إعادة تنظيم دائرة الحسابات والخزينة
Word count: 98
id: https://qanoon.om/p/1974/rd1974006/
Text:
نحن قابوس بن سعيد سلطان عمان نظرا لما تقتضيه المصلحة العامة فقد قررنا إعادة تنظيم دائرة الحسابات والخزينة. رسمنا بما هو آت المادة ١ تسمى هذه الدائرة بدائرة المالية بدلا من دائرة الحسابات والخزينة. المادة ٢ تقع مسؤولة هذه الدائرة على مستشار الشؤون المالية. المادة ٣ تستقل دائرة تدقيق الحسابات عن دائرة المالية وتكون تحت مسؤولية وزارة شؤون الديوان السلطاني. المادة ٤ ينشر هذا المرسوم في الجريدة الرسمية. صدر في: ١٩ محرم ١٣٩٤هـ الموافق: ١٢ فبراير ١٩٧٤م قابوس بن سعيد سلطان عمان نشر هذا المرسوم في عدد الجريدة الرسمية رقم (٥٠) الصادر في ٢ / ٣ / ١٩٧٤م
-------------------
Title: مرسوم سلطاني رقم ١٧ / ٧٤ بالتصديق على اتفاقية تأسيس البنك العربي للتنمية الاقتصادية في أفريقيا والنظام الأساسي المرافق لها
Word count: 92
id: https://qanoon.om/p/1974/rd1974017/
Text:
نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على اتفاقية تأسيس 

In [6]:
df_clean = df_clean[(df_clean['text_word_count'] >= 70)]
print("\nآمار طول متن:")
print(df_clean['text_word_count'].describe())


آمار طول متن:
count      7522.000000
mean        960.162191
std        3574.620576
min          71.000000
25%         125.000000
50%         195.000000
75%         692.000000
max      220969.000000
Name: text_word_count, dtype: float64


In [7]:
texts_q1 = df_clean[df_clean['text_word_count'] < 132]

for i, row in texts_q1.iterrows():
    print("\n-----------------")
    print("link:", row['canonical_link'])
    print("Title:", row['title'])
    print("Word count:", row['text_word_count'])
    print("Text :")
    print(row['text'])

print("-----------")
print("total news :", len(texts_q1))



-----------------
link: https://qanoon.om/p/1974/rd1974006/
Title: مرسوم سلطاني رقم ٦ / ٧٤ بخصوص إعادة تنظيم دائرة الحسابات والخزينة
Word count: 98
Text :
نحن قابوس بن سعيد سلطان عمان نظرا لما تقتضيه المصلحة العامة فقد قررنا إعادة تنظيم دائرة الحسابات والخزينة. رسمنا بما هو آت المادة ١ تسمى هذه الدائرة بدائرة المالية بدلا من دائرة الحسابات والخزينة. المادة ٢ تقع مسؤولة هذه الدائرة على مستشار الشؤون المالية. المادة ٣ تستقل دائرة تدقيق الحسابات عن دائرة المالية وتكون تحت مسؤولية وزارة شؤون الديوان السلطاني. المادة ٤ ينشر هذا المرسوم في الجريدة الرسمية. صدر في: ١٩ محرم ١٣٩٤هـ الموافق: ١٢ فبراير ١٩٧٤م قابوس بن سعيد سلطان عمان نشر هذا المرسوم في عدد الجريدة الرسمية رقم (٥٠) الصادر في ٢ / ٣ / ١٩٧٤م

-----------------
link: https://qanoon.om/p/1974/rd1974011/
Title: مرسوم سلطاني رقم ١١ / ٧٤ بتعديل الفقرة (٣) من المادة (٢) من قانون الجنسية العماني رقم (١) لسنة ١٩٧٢
Word count: 126
Text :
بعد الاطلاع على قانون الجنسية العمانية رقم (١) لسنة ١٩٧٢ وبناء على ما عرضه معالي وزير الداخلية والعدل على 

In [8]:
from langdetect import detect

def is_english(text):
    try:
        # تشخیص زبان متن
        return detect(text) == 'en'
    except:
        # برای متن‌های خالی یا عددی که قابل تشخیص نیستند
        return False

# پیدا کردن سطرهای انگلیسی
# ماسک بولی
english_mask = df_clean['text'].apply(is_english)

print(f"تعداد سطرهای انگلیسی: {english_mask.sum()}")

# حذف سطرهای انگلیسی
df_clean = df_clean.loc[~english_mask].reset_index(drop=True)

تعداد سطرهای انگلیسی: 30


In [9]:
print(df_clean['text_word_count'].describe())

count      7492.000000
mean        939.981046
std        3545.746924
min          71.000000
25%         124.750000
50%         194.000000
75%         680.000000
max      220969.000000
Name: text_word_count, dtype: float64


In [10]:
n_samples = 200
random_samples = df_clean.sample(n=n_samples)

for i, row in random_samples.iterrows():
    print("Title:", row['title'])

Title: مرسوم سلطاني رقم ٦١ / ٢٠١٠ بمنح الدرجة الخاصة
Title: مرسوم سلطاني رقم ١٦ / ٢٠١٢ بتعديل بعض أحكام المرسوم السلطاني رقم ١٠ / ٢٠١٠ بتعيين أعضاء لجنة حقوق الإنسان
Title: مرسوم سلطاني رقم ٣٧ / ٧٩ بتجديد عضوية الاستاذ محمود محمد مراد في مجلس محافظي البنك المركزي العماني
Title: مرسوم سلطاني رقم ٣٣ / ٢٠٢٥ بإجراء تنقلات وتعيينات في السلك الدبلوماسي
Title: مرسوم سلطاني رقم ٦٠ / ٨٨ بإجازة اتفاقية التعاون الاقتصادي والفني بين حكومة سلطنة عمان والاتحاد الاقتصادي البلجيكي اللوكسمبورجي والتصديق عليها
Title: مرسوم سلطاني رقم ٥٨ / ٢٠١١ بتعيين عضوين في مجلس الدولة
Title: مرسوم سلطاني رقم ٨١ / ٩٧ بتعيين أعضاء المجلس البلدي لبلدية مسقط
Title: هيئة الخدمات المالية: قرار رقم خ / ٢ / ٢٠٢٥ باعتماد نماذج معايير التقارير المالية الدولية
Title: مرسوم سلطاني رقم ٢٠ / ٩٢ بالتصديق على اتفاقية تجنب الازدواج الضريبي على الدخل الناشئ عن نشاط النقل الجوي الدولي بين حكومة سلطنة عمان وحكومة جمهورية تنزانيا المتحدة
Title: الجريدة الرسمية العدد ١٥٣٩
Title: مرسوم سلطاني رقم ٤٩ / ٨١ بنقل وكيل شؤون الأراضي في وزارة شؤو

In [11]:
title_mask = df_clean[df_clean['title_word_count'] <= 11]

n_samples = 100
random_samples = title_mask.sample(n=n_samples)

for i, row in random_samples.iterrows():
    print("Title:", row['title'])


print("-----------")
print("total news :", len(title_mask))

Title: وزارة الخارجية: استدراك
Title: مرسوم سلطاني رقم ٤٩ / ٢٠١٢ بتعيين أمين عام لمجلس التعليم
Title: مرسوم سلطاني رقم ٢١ / ٨٧ بتعيين نائبين لوالي ظفار
Title: مرسوم سلطاني رقم ٥٦ / ٢٠٠٩ بمنح الجنسية العمانية
Title: الجريدة الرسمية العدد ١٤٦٤
Title: مرسوم سلطاني رقم ٦ / ٩٠ بإجراء تعديل في التشكيل الوزاري
Title: وزارة العدل والشؤون القانونية: فتوى رقم ٢٢٢٧٥٩٣٥٤
Title: وزارة العدل والشؤون القانونية: استدراك
Title: مرسوم سلطاني رقم ٩٤ / ٢٠٠٧ بإجراء تعديل وزاري
Title: مرسوم سلطاني رقم ٦٩ / ٩٥ بإجراء تعديل في التشكيل الوزاري
Title: الجريدة الرسمية العدد ١٦٢٣
Title: مرسوم سلطاني رقم ١٣ / ٢٠١١ بإجراء تعديل في التشكيل الوزاري
Title: مرسوم سلطاني رقم ٣٥ / ٢٠٠٣ بإصدار قانون العمل
Title: مرسوم سلطاني رقم ٢٤ / ٩٦ بمنح الجنسية العمانية
Title: مرسوم سلطاني رقم ٤٩ / ٩٥ بإعادة تشكيل مجلس الشؤون المالية
Title: مرسوم سلطاني رقم ٨٠ / ٧٧ بمنح الجنسية العمانية لبعض الأشخاص
Title: الجريدة الرسمية العدد ١٤٥٧
Title: مرسوم سلطاني رقم ٥٧ / ٢٠٢١ بإصدار نظام جهاز الاستثمار العماني
Title: مرسوم سلطاني رقم ٦٤ / ٢٠٢٢

In [12]:
import re
import pandas as pd

# =========================
# 1. CLEANING & PRE-PROCESSING
# =========================

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # حفاظت از نوع سندهایی که دو نقطه دارند
    protected_starts = [
        "مرسوم", "أمر", "امر", "قرار", "قانون", "بيان", "منشور", "تعميم",
        "لجنة", "إعلان", "اعلان", "ديوان"
    ]

    match = re.match(r"^([^:]+)[:：]\s*(.*)", text)
    if match:
        prefix = match.group(1).strip()
        content = match.group(2).strip()
        is_protected = any(doc in prefix for doc in protected_starts)
        if is_protected:
            text = f"{prefix} {content}"
        else:
            text = content # حذف نام وزارتخانه

    # حذف نویزهای عددی و پرانتزها
    text = re.sub(r"(رقم|عدد)?\s*[\d٠-٩]+[\s\\/\-]+[\d٠-٩]+", " ", text)
    text = re.sub(r"(العدد|رقم)\s*[\d٠-٩]+", " ", text)
    text = re.sub(r"\([\d٠-٩]+\)", " ", text)

    # یکسان‌سازی کاراکترها
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى\b", "ي", text) # یکسان‌سازی ی آخر

    # حذف فاصله‌های اضافی
    text = re.sub(r"\s+", " ", text).strip()

    return text

def normalize_legal_action(text: str) -> str:
    """
    توسعه گسترده دایره لغات برای پوشش موارد خاص (تخویل، تطبیق، استحداث و...)
    """
    # پیشوندها: بـ، و، لـ، فـ، کـ + (الـ) اختیاری
    prefix = r"\b(?:ب|و|ل|ف|ك)?(?:ال)?"

    actions_map = {
        # --- اعطای اختیار و صلاحیت (گروه جدید مهم) ---
        f"{prefix}تخويل\\b": "منح",      # تخويل صفة الضبطية -> منح
        f"{prefix}تولي\\b": "تكليف",      # تولي السيد... -> تكليف
        f"{prefix}تكليف\\b": "تكليف",
        f"{prefix}اسناد\\b": "تفويض",     # اسناد الاختصاصات
        f"{prefix}تسيير\\b": "تكليف",     # تسيير اعمال
        f"{prefix}اختيار\\b": "تعيين",    # اختيار من يتولى
        f"{prefix}انابة\\b": "تفويض",     # ضوابط الانابة

        # --- فنی / اجرایی / مالی ---
        f"{prefix}تطبيق\\b": "تطبيق",     # تطبيق المواصفات
        f"{prefix}العمل\\b": "تطبيق",     # العمل بمواصفات
        f"{prefix}استحداث\\b": "انشاء",   # استحداث مديريات
        f"{prefix}اضافة\\b": "تعديل",     # اضافة عضو/مادة -> نوعی اصلاح است
        f"{prefix}ضم\\b": "تعيين",        # ضم عضو -> تعيين
        f"{prefix}تحصيل\\b": "فرض",       # تحصيل رسم -> فرض
        f"{prefix}فرض\\b": "فرض",         # فرض رسم
        f"{prefix}استقطاع\\b": "فرض",     # استقطاع نسبة
        f"{prefix}الزام\\b": "الزام",     # الزام اصحاب المنشات
        f"{prefix}سماح\\b": "تصريح",      # السماح لمواطني...
        f"{prefix}ترخيص\\b": "تصريح",     # بالترخيص
        f"{prefix}ادراج\\b": "ادراج",     # ادراج في القائمة (تروریسم)
        f"{prefix}تنسيق\\b": "تنظيم",     # تنسيق العمل

        # --- موارد استاندارد (قبلی) ---
        f"{prefix}تصديق\\b": "تصديق",
        f"{prefix}مصادقة\\b": "تصديق",
        f"{prefix}اجازة\\b": "تصديق",
        f"{prefix}موافقة\\b": "تصديق",
        f"{prefix}اعتماد\\b": "اعتماد",
        f"{prefix}اصدار\\b": "اصدار",
        f"{prefix}تعديل\\b": "تعديل",
        f"{prefix}تعديلات\\b": "تعديل",
        f"{prefix}الغاء\\b": "الغاء",
        f"{prefix}تنظيم\\b": "تنظيم",
        f"{prefix}انشاء\\b": "انشاء",
        f"{prefix}تاسيس\\b": "انشاء",
        f"{prefix}تشكيل\\b": "تشكيل",
        f"{prefix}اعادة تشكيل\\b": "اعادة تشكيل",
        f"{prefix}تعيين\\b": "تعيين",
        f"{prefix}تسمية\\b": "تعيين",
        f"{prefix}نقل\\b": "نقل",
        f"{prefix}ترقية\\b": "ترقية",
        f"{prefix}ترقيات\\b": "ترقية",
        f"{prefix}منح\\b": "منح",
        f"{prefix}رد\\b": "رد",
        f"{prefix}تجريد\\b": "تجريد",
        f"{prefix}سحب\\b": "تجريد",
        f"{prefix}تنازل\\b": "تنازل",     # تنازل عن الجنسية
        f"{prefix}احالة\\b": "احالة",     # احالة الى التقاعد
        f"{prefix}استبدال\\b": "استبدال", # استبدال عضو
        f"{prefix}تفويض\\b": "تفويض",
        f"{prefix}انضمام\\b": "انضمام",
        f"{prefix}تحديد\\b": "تحديد",
        f"{prefix}تخصيص\\b": "تخصيص",
        f"{prefix}حظر\\b": "حظر",
        f"{prefix}منع\\b": "حظر",
        f"{prefix}وقف\\b": "وقف",
        f"{prefix}ايقاف\\b": "وقف",
        f"{prefix}حل\\b": "حل",
        f"{prefix}دمج\\b": "دمج",
        f"{prefix}نشر\\b": "نشر",
        f"{prefix}زيادة\\b": "زيادة",
        f"{prefix}تخفيض\\b": "تخفيض",
        f"{prefix}رفع\\b": "رفع",
        f"{prefix}اشهار\\b": "اشهار",
        f"{prefix}استثناء\\b": "استثناء",
        f"{prefix}اعفاء\\b": "اعفاء",
        f"{prefix}تجديد\\b": "تجديد",
        f"{prefix}استمرار\\b": "تجديد",
        f"{prefix}تمديد\\b": "تمديد",
        f"{prefix}تصنيف\\b": "تصنيف",
        f"{prefix}اعتبار\\b": "اعتبار",
        f"{prefix}توقيع\\b": "توقيع",
        f"{prefix}تقرير\\b": "تقرير",
        f"{prefix}بدء\\b": "بدء",
        f"{prefix}تحويل\\b": "تحويل",
        f"{prefix}تغيير\\b": "تغيير",
        f"{prefix}تصحيح\\b": "تصحيح",
        f"{prefix}تفسير\\b": "تفسير",
        f"{prefix}سريان\\b": "سريان",
        f"{prefix}تعويض\\b": "تعويض",

        # --- هندل کردن جملات اسمی (Implicit Actions) ---
        # وقتی تیتر با اسم شروع می‌شود، فعل متناظر را جایگزین می‌کنیم
        f"^{prefix}الاحكام\\b": "تنظيم الاحكام",
        f"^{prefix}القانون\\b": "اصدار القانون",
        f"^{prefix}ترخيص\\b": "منح ترخيص",
        f"^{prefix}سياسة\\b": "اعتماد سياسة", # السياسة العامة
        f"^{prefix}بيان\\b": "اصدار بيان",

        # --- نویزها ---
        f"{prefix}اجراء\\b": "",
        f"{prefix}بخصوص\\b": "تنظيم",
        r"\bخاص\s+(?:ب|ل)?": "تنظيم ",
    }

    for pattern, repl in actions_map.items():
        text = re.sub(pattern, repl, text)

    return text

# =========================
# 2. LISTS
# =========================

DOC_TYPES = [
    "مرسوم سلطاني", "مرسوم",
    "امر سامي", "امر",
    "قرار وزاري", "قرار اداري", "قرار ديواني", "قرار", "ترخيص",
    "قانون", "لائحة", "نظام", "تعميم", "منشور", "بيان", "اعلان",
    "لجنة" # برای کمیته‌های خاص
]

LEGAL_ACTIONS = [
    "تقرير صفة المنفعة العامة",
    "اعادة تشكيل", "تحديد اختصاصات",
    "تصديق", "اصدار", "تعديل", "الغاء", "تنظيم", "تغيير", "تصحيح", "تفسير",
    "انشاء", "تشكيل", "تعيين", "نقل", "ترقية", "منح",
    "رد", "تجريد", "تنازل", "احالة", "اعتماد", "تفويض", "تكليف",
    "انضمام", "تحديد", "تخصيص", "حظر", "وقف", "حل", "دمج",
    "نشر", "زيادة", "تخفيض", "رفع", "اشهار", "استثناء",
    "اعفاء", "تجديد", "تمديد", "تصنيف", "اعتبار",
    "توقيع", "تقرير", "بدء", "سريان", "تصريح",
    "الحاق", "تحويل", "فرض", "الزام", "ادراج",
    "استقطاع", "استبدال", "تعويض", "تطبيق"
]

# =========================
# 3. EXTRACTION LOGIC
# =========================

def extract_title_semantic(title_raw: str):
    clean = clean_text(title_raw)

    # فیلتر نویز مطلق
    if "الجريدة الرسمية" in clean or re.search(r"فتوى\s+رقم\s*$", clean):
        return None
    if clean.strip() in ["استدراك", "استدراك في شأن النشيد الوطني"]: # استدراک‌های بدون محتوا
        return None

    # 1. Detect Doc Type
    doc_type = None

    # هندل کردن "اللجنة العليا" و "لجنة العقوبات"
    if "اللجنة" in clean and "قرار" in clean:
         # استخراج نام کمیته به عنوان نوع سند
         match_comm = re.match(r"(.*لجنة.*?):?\s*قرار", clean)
         if match_comm:
             doc_type = "قرار لجنة"
         else:
             doc_type = "قرار"

    if not doc_type:
        sorted_doc_types = sorted(DOC_TYPES, key=len, reverse=True)
        for dt in sorted_doc_types:
            if clean.startswith(dt):
                doc_type = dt
                clean = clean[len(dt):].strip()

                # Normalize doc type
                if "قرار" in dt: doc_type = "قرار"
                if "مرسوم" in dt: doc_type = "مرسوم"
                if "امر" in dt: doc_type = "امر"
                break

    # Implicit Doc Types
    if not doc_type:
        if clean.startswith("قانون"): doc_type = "قانون"
        elif clean.startswith("ترخيص"): doc_type = "ترخيص"
        elif clean.startswith("بدمج") or clean.startswith("باصدار"): doc_type = "سند عام"

    # 2. Normalize Actions
    clean_normalized = normalize_legal_action(clean)

    # 3. Find Action (First occurrence priority)
    best_action = None
    best_index = float('inf')
    best_end_index = 0

    for act in LEGAL_ACTIONS:
        pattern = r"\b" + re.escape(act) + r"\b"
        match = re.search(pattern, clean_normalized)
        if match:
            # اولویت با فعلی است که در ابتدای جمله آمده باشد
            if match.start() < best_index:
                best_index = match.start()
                best_action = act
                best_end_index = match.end()

    # 4. Fallback Logic
    if not best_action:
        if "في شان" in clean_normalized or "بشان" in clean_normalized:
            best_action = "تنظيم"
            match = re.search(r"(في|ب)\s*شان", clean_normalized)
            if match: best_end_index = match.end()

    if not best_action and doc_type in ["قانون", "نظام", "لائحة"]:
        best_action = "اصدار"
        best_end_index = 0

    # 5. Extract Subject
    if best_action:
        subject = clean_normalized[best_end_index:].strip()
    else:
        subject = clean_normalized

    # Cleanup Subject
    subject = re.sub(r"^\s*(على|في|ب|عن|الى|ل|من)\s+", "", subject)

    # Validation
    if len(subject) < 3 or "فتوى رقم" in clean:
        return None

    if not doc_type and not best_action:
        return None

    # اگر نوع سند پیدا نشد اما فعل داریم، فرض می‌کنیم سند عمومی است
    if not doc_type and best_action:
        doc_type = "سند"

    return f"{doc_type} | {best_action} | {subject}"

# Example Usage:
df_clean["title_semantic"] = df_clean['title'].apply(extract_title_semantic)

failed_mask = df_clean['title_semantic'].isna() | (df_clean['title_semantic'].str.strip() == '')
print('تعداد نمونه‌هایی که title_semantic استخراج نشده:')
print(failed_mask.sum())
failed_df = df_clean.loc[failed_mask]

تعداد نمونه‌هایی که title_semantic استخراج نشده:
475


In [13]:
n_samples = 200
random_samples = df_clean.sample(n=n_samples)

for i, row in random_samples.iterrows():
    print("title_raw:", row['title'])
    print("Title_embd:", row['title_semantic'])
    print("--------------")

title_raw: مرسوم سلطاني رقم ٣١ / ٩٥ بتعيين سفراء غير مقيمين
Title_embd: مرسوم | تعيين | سفراء غير مقيمين
--------------
title_raw: مرسوم سلطاني رقم ٤٤ / ٢٠١٠ بتعيين سفير غير مقيم
Title_embd: مرسوم | تعيين | سفير غير مقيم
--------------
title_raw: مرسوم سلطاني رقم ٢٤ / ٨٧ بتعيين رئيس للإدارة واللوازم بوزارة الدفاع
Title_embd: مرسوم | تعيين | رئيس للادارة واللوازم بوزارة الدفاع
--------------
title_raw: وزارة التنمية الاجتماعية: قرار وزاري رقم ٢٣٥ / ٢٠١٤ بإصدار اللائحة التنظيمية لصرف الأجهزة التعويضية والوسائل المساعدة
Title_embd: قرار | اصدار | اللائحة التنظيمية لصرف الاجهزة التعويضية والوسائل المساعدة
--------------
title_raw: هيئة حماية المستهلك: قرار رقم ١ / ٢٠٢٣ بتعديل بعض أحكام اللائحة التنفيذية لقانون حماية المستهلك
Title_embd: قرار | تعديل | بعض احكام اللائحة التنفيذية لقانون حماية المستهلك
--------------
title_raw: وزارة التعليم العالي والبحث العلمي والابتكار: قرار وزاري رقم ٤٠ / ٢٠٢١ بإصدار لائحة تنظيم التدريب الخاص
Title_embd: قرار | اصدار | لائحة تنظيم التدريب الخاص
---------

In [14]:
n_samples = min(500, len(failed_df))
random_samples = failed_df.sample(n=n_samples, random_state=42)

for _, row in random_samples.iterrows():
    if len(str(row['title']).split()) > 10:  # تعداد کلمات بیشتر از 10
        print("title:", row['title'])


title: مرسوم سلطاني رقم ٥٩ / ٧٩ بنقل تبعية ديوان التشريع والجريدة الرسمية
title: مرسوم سلطاني رقم ٤٥ / ٧٨ بإلحاق المديرية العامة للأشغال العامة بوزارة الشؤون الاجتماعية والعمل
title: وزارة الصحة: قرار وزاري رقم ٤١ / ٩١ بشأن الفحوصات الطبية للكشف عن الأمراض المعدية للوافدين للسلطنة بغرض العمل
title: مرسوم سلطاني رقم ٨٦ / ٢٠٢٥ بتعديل بعض أحكام قانون الجريدة الرسمية


In [15]:
import re
import pandas as pd

# اصلاحات انجام شده در پترن:
# 1. اضافه کردن (?:في|بتاريخ) برای پوشش هر دو حالت.
# 2. اضافه کردن \d+ بلافاصله بعد از دو نقطه/فاصله. این مهمترین بخش است که
#    باعث می‌شود عباراتی مثل "صدر في سلطنة" (که بعدش عدد نیست) نادیده گرفته شوند.
FOOTER_PATTERN = re.compile(
   r'(صدر\s+(?:في|بتاريخ)\s*[:]?\s*\d+[\s\S]*?(?:الصادر|الصادرة)\s+في\s+.*?\d{4}\s*[مهـ]?\.?)'
)

def extract_and_remove_footer(text):
    if not isinstance(text, str):
        return text, None

    # استفاده از finditer برای پیدا کردن تمام موارد ممکن
    matches = list(FOOTER_PATTERN.finditer(text))

    if matches:
        # همیشه آخرین مورد پیدا شده را انتخاب می‌کنیم (چون فوتر در انتهای متن است)
        last_match = matches[-1]
        footer = last_match.group(1).strip()

        # متن را بر اساس مکان شروع فوتر برش می‌زنیم تا مطمئن شویم
        # اگر مورد مشابهی در وسط متن بود اشتباهاً حذف نشود.
        start_index = last_match.start()
        clean_text = text[:start_index].strip()

        return clean_text, footer

    return text, None



In [16]:
df_clean[['text', 'footer']] = df_clean['text'].apply(
    lambda x: pd.Series(extract_and_remove_footer(x))
)
failed_mask = df_clean['footer'].isna() | (df_clean['footer'].str.strip() == '')
print('تعداد نمونه‌هایی که footer استخراج نشده:')
print(failed_mask.sum())


تعداد نمونه‌هایی که footer استخراج نشده:
729


In [17]:
failed_df = df_clean.loc[failed_mask]

n_samples = min(500, len(failed_df))
random_samples = failed_df.sample(n=n_samples, random_state=42)

for _, row in random_samples.head(500).iterrows():
    print(row['text'][-400:])
    print("="*80)


ركة ثوابت الشمال للتجارة والمقاولات – تضامنية. إعلان عن انتهاء أعمال التصفية لشركة مشاريع صحم الشامخة الوطنية – تضامنية. إعلان عن انتهاء أعمال التصفية لشركة رموز الاتحاد الخليجي للمقاولات ش.م.م. إعلان عن انتهاء أعمال التصفية لشركة مجان للغوص والخدمات الفنية البحرية ش.م.م. إعلان عن انتهاء أعمال التصفية لشركة المعدن النادر ش.م.م. إعلان عن انتهاء أعمال التصفية لشركة أسطول مسقط للمشاريع الحديثة ش.م.م.
 المادة (٣٩) من قانون تنظيم وتخصيص قطاع الكهرباء والمياه المرتبطة به المشار إليه بند جديد برقم (ط مكررا)، يكون نصه كالآتي: “ط مكررا – الأحكام والشروط والإجراءات والضوابط الفنية المتعلقة بإنتاج وبيع الكهرباء باستخدام مصادر الطاقة المتجددة”. المادة (٤) تلغى عبارة “والهيئة العامة للكهرباء والمياه” الواردة في البند (٢١) من المادة (٢٢) من قانون تنظيم وتخصيص قطاع الكهرباء والمياه المرتبطة به المشار إليه
الالتزام بجميع التدابير الاحترازية الموضوعة من قبل الجهات المختصة وأهمها لبس الكمامة والتباعد الاجتماعي سواء في أماكن السكن أو العمل والأماكن العامة، وعدم التهاون في الالتزام بها. ونظرا لتسارع المست

In [18]:
n_samples = 715
random_samples = df_clean.sample(n=n_samples)

for i, row in random_samples.head(715).iterrows():
    print("------")
    print(row['footer'])

------
صدر في: ٦ من ربيع الثاني سنة ١٤٢٥ هـ الموافق: ٢٦ من مايو سنة ٢٠٠٤ م قابوس بن سعيد سلطان عمان نشر هذا المرسوم في عدد الجريدة الرسمية رقم (٧٦٨) الصادر في ١ / ٦ / ٢٠٠٤م
------
صدر في: ٢١ من شعبان سنة ١٤١٧هـ الموافق: ١ من يناير سنة ١٩٩٧م قابوس بن سعيد سلطان عمان نشر هذا المرسوم في عدد الجريدة الرسمية رقم (٥٩٠) الصادر في ١ / ١ / ١٩٩٧م.
------
صدر في: ١٢ من صفر سنة ١٤٢٧هـ الموافق: ١٢ من مارس سنة ٢٠٠٦م قابوس بن سعيد سلطان عمان نشر هذا المرسوم في عدد الجريدة الرسمية رقم (٨١١) الصادر في ١٥ / ٣ / ٢٠٠٦م
------
صدر في: ١٦ من جمادى الأولى سنة ١٤٤٠هـ الموافق: ٢٣ من يناير سنة ٢٠١٩م قابوس بن سعيد سلطان عمان نشر هذا المرسوم في عدد الجريدة الرسمية رقم (١٢٧٨) الصادر في ٢٧ / ١ / ٢٠١٩م.
------
صدر في: ١١ محرم سنة ١٤٠٢هـ الموافق: ٩ نوفمبر سنة ١٩٨١م قابوس بن سعيد سلطان عمان نشر هذا المرسوم في عدد الجريدة الرسمية رقم (٢٢٩) الصادر في ١٥ / ١١ / ١٩٨١م
------
صدر في: ٨ من جمادى الأولى سنة ١٤٢٦هـ الموافق: ١٥ من يونيو سنة ٢٠٠٥م قابوس بن سعيد سلطان عمان نشر هذا المرسوم في عدد الجريدة الرسمية رقم (٧٩٤) الصادر 

In [19]:
# اضافه کردن ستون طول متن و تعداد کلمات
df_clean['text_word_count'] = df_clean['text'].apply(lambda x: len(x.split()))

print("\nآمار طول متن:")
print(df_clean['text_word_count'].describe())


آمار طول متن:
count     7492.000000
mean       263.253604
std        932.541465
min         27.000000
25%         80.000000
50%        104.000000
75%        155.000000
max      28623.000000
Name: text_word_count, dtype: float64


In [20]:
n_samples = 500
random_samples = df_clean.sample(n=n_samples)

for i, row in random_samples.iterrows():
    print('=' * 80)
    print(f'Index: {i}')
    print(row['text'][-300:])

Index: 3416
التي يقدرها الاستثناء من هذا الشرط لمن أمضى بنجاح ثمان سنوات دراسية. البند (٧) أن يخضع لنفس المتطلبات الدراسية اللازمة للحصول على المؤهل في التعليم دون الجامعي والجامعي والدراسات العليا، وأن تكون الدراسة نظامية. المادة الثانية ينشر هذا القرار في الجريدة الرسمية، ويعمل به من اليوم التالي لتاريخ نشره.
Index: 652
٤ تحل اللجنة العليا لإعداد الترشيحات النهائية لنيل أوسمة الاستحقاق المشكلة بمقتضى المرسوم السلطاني رقم ٥٢ / ٧٦. مادة ٥ على الجهات المعنية في السلطنة معاونة المجلس الأعلى لرعاية الشباب في تنفيذ هذا المرسوم كل في حدود اختصاصه. مادة ٦ ينشر هذا المرسوم في الجريدة الرسمية ويعمل به اعتبارا من تاريخ صدوره.
Index: 287
يس بلدية العاصمة إلى وظيفة وكيل وزارة لشؤون البلديات بوزارة شؤون الأراضي والبلديات اعتبارا من ١ /٦ / ١٩٧٨. المادة ٣ على وزير شؤون الأراضي والبلديات إعادة ترتيب الهيكل التنظيمي للوزارة وتحديد اختصاصات الوظائف فيها طبقا لهذا المرسوم. المادة ٤ ينشر هذا المرسوم في الجريدة الرسمية ويعمل به من تاريخ نشره.
Index: 5239
ارات، تسري على الوحدة القوانين والنظم المطبقة على د

In [21]:
df_clean.iloc[10]

json_link                 https://qanoon.om/wp-json/wp/v2/posts/19879
short_link                                 https://qanoon.om/?p=19879
canonical_link                https://qanoon.om/p/1972/legacy1973005/
date                                              1972-12-28 00:00:00
title                                         قانون الشرطة رقم ٥ / ٧٣
text                نحن قابوس بن سعيد، سلطان عمان، أصدرنا القانون ...
html                <p><a class="pdf-link" href="http://data.qanoo...
error                                                             NaN
text_word_count                                                  6012
title_word_count                                                    6
title_semantic                                 قانون | اصدار | الشرطة
footer              صدر في :٢٢ ذي القعدة ١٣٩٢هـ الموافق ٢٨ ديسمبر ...
Name: 10, dtype: object

In [22]:
n_samples = 10
random_samples = df_clean.sample(n=n_samples)

for i, row in random_samples.iterrows():
    #print('=' * 80)
    print('-' * 10)
    #print(f'link: ',row['canonical_link'])
    print(f'Index: {i}')
    print(row['text'][:800])

----------
Index: 1179
نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على المرسوم السلطاني رقم ٢٦ / ٧٥ بإصدار قانون تنظيم الجهاز الإداري للدولة وتعديلاته، وعلى القانون رقم ٤ / ٧٤ في شأن الشركات التجارية، وعلى المرسوم السلطاني رقم ٤ / ٧٤ بإصدار قانون الحرف الأجنبية واستثمار الرأسمال الأجنبي، وعلى المرسوم السلطاني رقم ٨٣ / ٨٠ في شأن الدعم المالي للقطاع الخاص في مجالات الزراعة والأسماك والصناعة والمعادن والمحاجر، وعلى المرسوم السلطاني رقم ٧٠ / ٨١ باعتماد اللائحة التنفيذية للمرسوم السلطاني رقم ٨٣ / ٨٠ في مجال الصناعة، وبناء على ما تقتضيه المصلحة العامة. رسمنا بما هو آت مادة (١) يعمل في شأن الدعم المالي للقطاع الخاص في مجالي الصناعة والسياحة بأحكام النظام المرافق. مادة (٢) يصدر وزير التجارة والصناعة اللوائح والقرارات التنفيذية اللازمة لتطبيق أحكام هذا النظام. مادة (٣) يلغى المرسوم السلطاني رقم ٧٠ / ٨١ المشار إليه، كما يلغى
----------
Index: 263
نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على المرسوم رقم ٢٦ / ٧٥ بإصدار قانون تنظيم الجهاز الإداري للدولة وتعديلاته، وعلى المرسوم رقم ٤٢ / ٧٤ بإصدار قانون

In [23]:
import re

def extract_articles(text):
    if not isinstance(text, str) or not text.strip():
        return None

    # ---------------------------------------------------------
    # 1. برش مقدمه (Preamble Slicing) - اصلاح شده
    # ---------------------------------------------------------
    # تغییر: جستجو را فقط به 1000 کاراکتر اول محدود می‌کنیم تا
    # اگر کلمه 'تقرر' در وسط متن (مثلاً داخل ماده 20) آمد، کل متن برید نشود.

    start_markers = [r'رسمنا\s+بما\s+هو\s+آت', r'\bتقرر\b', r'\bقرر\b']
    processing_text = text

    # فقط در بخش ابتدایی متن دنبال این کلمات می‌گردیم
    preamble_limit = 1000
    search_scope = text[:preamble_limit]

    best_start_index = 0
    for marker in start_markers:
        match = re.search(marker, search_scope)
        if match:
            # اگر پیدا شد، متن را از آنجا به بعد در نظر می‌گیریم
            best_start_index = match.end()
            break

    if best_start_index > 0:
        processing_text = text[best_start_index:]

    # ---------------------------------------------------------
    # 2. پترن‌های اعداد و حروف
    # ---------------------------------------------------------
    number_words = [
        r'الأولى', r'الثانية', r'الثالثة', r'الرابعة', r'الخامسة', r'السادسة',
        r'السابعة', r'الثامنة', r'التاسعة', r'العاشرة', r'الحادية', r'الثانية',
        r'أولى', r'ثانية', r'ثالثة', r'رابعة', r'خامسة', r'سادسة',
        r'سابعة', r'ثامنة', r'تاسعة', r'عاشرة', r'حادية', r'ثاني',
        r'واحد', r'اثنان', r'ثلاثة', r'أربعة', r'خمسة', r'ستة',
        r'سبعة', r'ثمانية', r'تسعة', r'عشر', r'عشرة',
        r'العشرون', r'الثلاثون', r'الأربعون', r'الخمسون'
    ]
    word_pattern = '|'.join(number_words)

    # اصلاح پترن عدد: پشتیبانی از پرانتزهای مختلف و فاصله‌های احتمالی
    # همچنین هندل کردن "مکرر" (مثلاً ماده 5 مکرر)
    digit_part = r'(?:\(\s*[\d\u06F0-\u06F9\u0660-\u0669]+\s*\)|[\d\u06F0-\u06F9\u0660-\u0669]+)'
    text_part = f'(?:(?:{word_pattern})(?:\s+(?:عشر|عشرة))?)'

    # پترن نهایی:
    # گروه 1: کل هدر (مثلاً: المادة 5 مکرر)
    # گروه 2: کلمه کلیدی (المادة یا مادة)
    # اضافه شدن (?:\s+مكرر)? برای مواد مکرر
    full_pattern = re.compile(
        r'((?:^|\s|[.،؛:])(المادة|مادة)\s+(?:' + digit_part + r'|' + text_part + r')(?:\s+مكرر)?)',
        re.MULTILINE
    )

    matches = list(full_pattern.finditer(processing_text))
    if not matches:
        return None

    # ---------------------------------------------------------
    # 3. فیلتر کردن هوشمند رفرنس‌ها (Reference Logic Improved)
    # ---------------------------------------------------------
    valid_matches = []

    # لیست کلماتی که اگر قبل از "ماده" بیایند، یعنی ارجاع است (نه شروع ماده)
    # اضافه شدن "من" برای حل مشکل نمونه 3590
    ref_keywords = [
        'في', 'من', 'حكم', 'نص', 'أحكام', 'بموجب', 'مقتضى',
        'خلال', 'تطبيق', 'وفق', 'أحكام', 'مراعاة'
    ]
    ref_pattern = re.compile(r'\b(' + '|'.join(ref_keywords) + r')\s*$', re.UNICODE)

    for m in matches:
        start_idx = m.start()
        # گروه 2 شامل کلمه "مادة" یا "المادة" است
        keyword = m.group(2)

        # بررسی متن قبل (Prefix)
        prefix = processing_text[max(0, start_idx-20):start_idx]
        # تمیز کردن فاصله ها برای بررسی راحت تر
        prefix_clean = re.sub(r'\s+', ' ', prefix).strip()

        # 1. اگر قبلش کلمات ارجاعی مثل "في"، "من"، "حكم" بود -> حذف
        if ref_pattern.search(prefix_clean):
            continue

        # 2. بررسی حروف چسبان (ب، ل، ك) - اصلاح شده
        # نکته مهم: این حروف معمولا فقط به "المادة" (با الف و لام) می‌چسبند.
        # اگر کلمه "مادة" (بدون ال) باشد، معمولاً هدر اصلی است.
        if keyword == 'المادة':
            # چک می‌کنیم کاراکتر چسبیده به شروع مچ چیست
            # m.start(2) ایندکس شروع کلمه "المادة" است
            word_start = m.start(2)
            if word_start > 0:
                char_before = processing_text[word_start - 1]
                # اگر قبلش ب، ل، یا ک بود
                if char_before in 'بلك':
                    # برای اطمینان چک میکنیم که قبل از این حرف چسبان، فاصله یا شروع خط باشد
                    # مثال: "وبالمادة" (حذف شود) اما "جبال المادة" (حذف نشود - گرچه بعید است)
                    if word_start == 1 or (word_start > 1 and processing_text[word_start-2] in ' \t\n\r.،'):
                        continue

        valid_matches.append(m)

    if not valid_matches:
        return None

    # ---------------------------------------------------------
    # 4. استخراج متن
    # ---------------------------------------------------------
    extracted_articles = []
    for i, m in enumerate(valid_matches):
        # تمیز کردن هدر (حذف کاراکترهای اضافی اول اگر regex گرفته باشد)
        header_raw = m.group(1).strip()
        # حذف کاراکترهای غیر حرفی از ابتدای هدر (مثل نقطه یا ویرگول که در گروه 1 ممکن است باشد)
        header = re.sub(r'^[^\w\u0600-\u06FF]+', '', header_raw)

        start = m.end()

        if i + 1 < len(valid_matches):
            end = valid_matches[i+1].start()
            # یک نکته ظریف: پترن regex ما ممکن است با \s یا کاراکتر قبل شروع شود
            # بنابراین start مچ بعدی ممکن است کمی عقب تر از خود کلمه "ماده" باشد.
            # برای تمیزی بیشتر بهتر است تا شروع گروه "ماده" ی بعدی ببریم ولی همین هم کفایت میکند.
        else:
            end = len(processing_text)

        content = processing_text[start:end].strip()

        # حذف علائم نگارشی ابتدای متن (: - .)
        content = re.sub(r'^[:\-–.]\s*', '', content)

        # حذف پاورقی‌های احتمالی که به عدد ماده چسبیده‌اند مثل [1]
        # اگر می‌خواهید پاورقی در متن بماند، این خط را حذف کنید.
        # اما معمولا [1] بعد از "ماده (4)" جزو متن ماده است نه هدر.

        extracted_articles.append(f"{header} : {content}")

    return '\n----------------\n'.join(extracted_articles)
df_clean['articles'] = df_clean['text'].apply(extract_articles)

In [24]:
n_samples = 20
random_samples = df_clean.sample(n=n_samples)

for i, row in random_samples.iterrows():
    # print(f'Index: {i}')
    # print(row['text'])
    # print('-----' * 80)
    # print("[link]:")
    # print(row['canonical_link'])
    # print("[title]:")
    # print(row['title'])
    # print("[title_embd]:")
    # print(row['title_semantic'])
    print("[full text]:")
    print(row['text'][:800])
    print("[articles]:")
    print(row['articles'])
    # print("[footer]:")
    # print(row['footer'])


[full text]:
نحن هيثم بن طارق سلطان عمان بعد الاطلاع على النظام الأساسي للدولة، وعلى المرسوم السلطاني رقم ٦٧ / ٢٠٠٣ بتطبيق قانون الجمارك الموحد لدول مجلس التعاون لدول الخليج العربية، وبعد العرض على مجلس عمان، وبناء على ما تقتضيه المصلحة العامة. رسمنا بما هو آت المادة الأولى يعمل بقانون ضريبة القيمة المضافة، المرفق. المادة الثانية يصدر رئيس جهاز الضرائب اللائحة التنفيذية للقانون المرفق خلال مدة لا تزيد على (٦) ستة أشهر من تاريخ العمل به، كما يصدر القرارات اللازمة لتنفيذ أحكامه. المادة الثالثة يلغى كل ما يخالف القانون المرفق، أو يتعارض مع أحكامه. المادة الرابعة ينشر هذا المرسوم في الجريدة الرسمية، ويعمل به بعد (١٨٠) مائة وثمانين يوما من تاريخ نشره.
[articles]:
المادة الأولى : يعمل بقانون ضريبة القيمة المضافة، المرفق.
----------------
المادة الثانية : يصدر رئيس جهاز الضرائب اللائحة التنفيذية للقانون المرفق خلال مدة لا تزيد على (٦) ستة أشهر من تاريخ العمل به، كما يصدر القرارات اللازمة لتنفيذ أحكامه.
----------------
المادة الثالثة : يلغى كل ما يخالف القانون المرفق، أو يتعارض مع أحكامه.
---

In [25]:
import pandas as pd

# فیلتر کردن سطرهایی که 'articles' NaN یا None است
none_articles_df = df_clean[df_clean['articles'].isna()]

# نمایش تعداد این سطرها
print(f"تعداد نمونه هایی که ماده‌ای استخراج نشده: {len(none_articles_df)}")

# نمایش ستون 'text' برای ۱۰ نمونه اول (می‌توانید تعداد را تغییر دهید)
for i, text in enumerate(none_articles_df['text'][:800].head(50)):
    print(text)
    print("="*80)

تعداد نمونه هایی که ماده‌ای استخراج نشده: 390
نحن قابوس بن سعيد سلطان عمان حيث إن البلاد بحاجة إلى قانون جزاء يرتب علاقات الأفراد فيما بينهم ويحدد واجباتهم تجاه المجتمع والحق العام فقد أصدرنا قانون الجزاء العماني. ينشر هذا القانون في ملحق الجريدة الرسمية ويعمل به من تاريخ نشره.
نحن قابوس بن سعيد سلطان عمان بناء على ما تقتضيه المصلحة العامة، أصدرنا قانون السجون المرفق. ينشر هذا القانون في الجريدة الرسمية ويعمل به من تاريخ النشر.
نحن قابوس بن سعيد سلطان عمان بناء على ما عرضه علينا مجلس الشؤون المالية، رسمنا بما هو آت أولا الموافقة على الموازنة العامة للدولة لعام ١٩٧٦ حسب الجداول المرفقة. ثانيا على جميع الوزارات والدوائر الحكومية تنفيذ ذلك كل فيما يخصه. ثالثا ينشر هذا المرسوم في الجريدة الرسمية ويعمل به من تاريخ نشره.
استنادا إلى القانون المالي الصادر بالمرسوم السلطاني رقم ٤٧ / ٩٨، وإلى المرسوم السلطاني رقم ٢ / ٢٠١١ بالتصديق على الميزانية العامة للدولة للسنة المالية ٢٠١١م، وبعد العرض على المقام السامي. تقرر مادة وحيدة ينشر في الجريدة الرسمية الحساب الختامي للدولة عن السنة المالية ٢٠١١م طب

In [26]:
import pandas as pd
none_articles_df = df_clean[df_clean['articles'].isna()]
print(f"تعداد نمونه هایی که ماده‌ای استخراج نشده: {len(none_articles_df)}")

df_clean = df_clean[df_clean['articles'].notna()].reset_index(drop=True)


تعداد نمونه هایی که ماده‌ای استخراج نشده: 390


In [27]:
df_clean.iloc[0]['articles']

'المادة ١ : يعتبر عمانيا حكما: ١ – من ولد في عمان أو خارجا من أب عماني. ٢ – من ولد في عمان من والدين مجهولين. ٣ – الولد غير الشرعي القاصر الذي تثبت بنوته إذا كان أحد والديه الذي ثبتت البنوة أولا بالنظر إليه عمانيا. إذا كان ثبوت البنوة بالنظر إلى الأب والأم ناتجا عن عقد واحد أو حكم واحد اتخذ الولد تابعية الأب إذا كان هذا الأب عمانيا. ٤ – من ولد في عمان أو خارجها من أم عمانية وكان مجهول الأب أو لم تثبت نسبته لأبيه شرعا أو كان أبوه لا جنسية له. ٥ – من ولد في عمان وجعل منها محل إقامته العادية وكان أبوه قد ولد فيها على أن لا يكون الأب حاملا أي جنسية عند تلك الولادة. ٦ – من ينتمي بأصله لعمان ولم يكتسب جنسية أخرى ولم يتقدم لاختيار الجنسية العمانية حسب الأنظمة المرعية.\n----------------\nالمادة ٢ : يجوز منح الجنسية العمانية لأجنبي إذا توفرت فيه الشروط التالية: ١ – أن يكون بالغا سن الرشد. ٢ – أن يتقدم بطلب خطي. ٣ – أن يسبق طلبه إقامة فعلية متواصلة وشرعية في عمان لمدة لا تقل عن عشر سنوات، أما إذا كان متزوجا بعمانية فتخفض مدة الإقامة إلى سنتين بعد الزواج. ٤ – أن يكون حسن السيرة والسلوك وأن يكون س

In [28]:
DELIM = "\n----------------\n"
df_clean['num_articles'] = df_clean['articles'].apply(
    lambda x: len(x.split(DELIM)) if isinstance(x, str) else 0
)
df_clean['word_count_articles'] = df_clean['articles'].apply(
    lambda x: len(x.split()) if isinstance(x, str) else 0
)

def avg_words_per_article(text):
    if not isinstance(text, str):
        return 0
    parts = text.split(DELIM)
    return sum(len(p.split()) for p in parts) // len(parts)

df_clean['avg_words_per_article'] = df_clean['articles'].apply(avg_words_per_article)


In [29]:
import re
import pandas as pd

def extract_preamble(text):
    if not isinstance(text, str) or not text.strip():
        return None

    # ---------------------------------------------------------
    # 1. تعریف پترن‌های ماده (مشابه قبل)
    # ---------------------------------------------------------
    number_words = [
        r'الأولى', r'الثانية', r'الثالثة', r'الرابعة', r'الخامسة', r'السادسة',
        r'السابعة', r'الثامنة', r'التاسعة', r'العاشرة', r'الحادية', r'الثانية',
        r'أولى', r'ثانية', r'ثالثة', r'رابعة', r'خامسة', r'سادسة',
        r'سابعة', r'ثامنة', r'تاسعة', r'عاشرة', r'حادية', r'ثاني',
        r'واحد', r'اثنان', r'ثلاثة', r'أربعة', r'خمسة', r'ستة',
        r'سبعة', r'ثمانية', r'تسعة', r'عشر', r'عشرة',
        r'العشرون', r'الثلاثون', r'الأربعون', r'الخمسون'
    ]
    word_pattern = '|'.join(number_words)
    digit_part = r'(?:\(\s*[\d\u06F0-\u06F9\u0660-\u0669]+\s*\)|[\d\u06F0-\u06F9\u0660-\u0669]+)'
    text_part = f'(?:(?:{word_pattern})(?:\s+(?:عشر|عشرة))?)'

    full_pattern = re.compile(
        r'((?:^|\s|[.،؛:])(المادة|مادة)\s+(?:' + digit_part + r'|' + text_part + r')(?:\s+مكرر)?)',
        re.MULTILINE
    )

    matches = list(full_pattern.finditer(text))

    # متغیر برای ذخیره متن اولیه استخراج شده
    raw_preamble = text.strip()

    # ---------------------------------------------------------
    # 2. پیدا کردن نقطه شروع اولین ماده معتبر
    # ---------------------------------------------------------
    if matches:
        ref_keywords = [
            'في', 'من', 'حكم', 'نص', 'أحكام', 'بموجب', 'مقتضى',
            'خلال', 'تطبيق', 'وفق', 'أحكام', 'مراعاة'
        ]
        ref_pattern = re.compile(r'\b(' + '|'.join(ref_keywords) + r')\s*$', re.UNICODE)

        split_index = -1

        for m in matches:
            start_idx = m.start()
            keyword = m.group(2)
            prefix = text[max(0, start_idx-20):start_idx]
            prefix_clean = re.sub(r'\s+', ' ', prefix).strip()

            if ref_pattern.search(prefix_clean):
                continue

            if keyword == 'المادة':
                word_start = m.start(2)
                if word_start > 0:
                    char_before = text[word_start - 1]
                    if char_before in 'بلك':
                        if word_start == 1 or (word_start > 1 and text[word_start-2] in ' \t\n\r.،'):
                            continue

            split_index = start_idx
            break

        if split_index != -1:
            raw_preamble = text[:split_index].strip()

    # ---------------------------------------------------------
    # 3. پاکسازی عبارات دستوری از انتهای مقدمه (بخش جدید)
    # ---------------------------------------------------------

    # لیست عباراتی که باید از انتهای متن حذف شوند
    removal_phrases = [
        r'رسمنا\s+بما\s+هو\s+آت',  # رسمنا بما هو آت
        r'قررنا',                  # قررنا
        r'فقد\s+قررنا',            # فقد قررنا
        r'فقد\s+تقرر\s+ما\s*يلي',  # فقد تقرر مايلي / ما يلي
        r'فقد\s+تقرر',             # فقد تقرر
        r'أصدرنا\s+المرسوم\s+الآتي', # أصدرنا المرسوم الآتي
        r'أصدرنا\s+القانون\s+الآتي', # أصدرنا القانون الآتي
        r'ما\s*يلي'                 # مايلي (اگر تنها مانده باشد)
    ]

    # ترکیب پترن‌ها با OR
    # \s*[:.\-]*\s*$ به معنی: "به همراه هر گونه فاصله، نقطه، دو نقطه یا خط تیره که در انتهای خط باشد"
    clean_pattern = r'(?:' + '|'.join(removal_phrases) + r')\s*[:.\-]*\s*$'

    # حذف عبارت پیدا شده از انتهای متن
    clean_preamble = re.sub(clean_pattern, '', raw_preamble, flags=re.MULTILINE)

    # ---------------------------------------------------------
    # 4. پاکسازی نهایی علائم نگارشی باقی‌مانده
    # ---------------------------------------------------------
    # ممکن است بعد از حذف عبارت بالا، یک نقطه یا دو نقطه از جمله قبل باقی مانده باشد
    # مثال: "... ١٩٧٤. قررنا" -> بعد از حذف قررنا -> "... ١٩٧٤."
    # این خط علائم نگارشی اضافی انتهای متن را حذف می‌کند
    clean_preamble = re.sub(r'\s*[:.\-،]+\s*$', '', clean_preamble)

    return clean_preamble.strip()

# اجرا روی دیتافریم
df_clean['PREAMBLE'] = df_clean['text'].apply(extract_preamble)


In [30]:

# random_state=42 برای تکرارپذیری (هر بار نتیجه یکسان بدهد)
samples = df_clean[['text', 'articles', 'num_articles', 'word_count_articles', 'avg_words_per_article']].sample(n=5, random_state=42)

# پرینت کردن هر نمونه (با جداکننده برای خوانایی)
for idx, row in samples.iterrows():
    print(f"نمونه {idx}:")
    print(f"Text: {row['text']}")
    print(f"Num Articles: {row['num_articles']}")
    print(f"Word Count Articles: {row['word_count_articles']}")
    print(f"avg_words_per_article: {row['avg_words_per_article']}")
    print(f"articles: {row['articles']}")
    print("-" * 80)  # جداکننده بلند برای خواندن بهتر

نمونه 6095:
Text: استنادا إلى المرسوم السلطاني رقم ٣٥ / ٢٠١٤ بإنشاء كلية العلوم الشرعية وإصدار نظامها، وإلى اللائحة الداخلية لكلية العلوم الشرعية الصادرة بالقرار رقم ٣٥ / ٢٠١٨، وبناء على ما تقتضيه المصلحة العامة. تقرر المادة الأولى يعمل بأحكام النظام الأكاديمي لكلية العلوم الشرعية، المرفق. المادة الثانية يلغى كل ما يخالف هذا القرار، أو يتعارض مع أحكامه. المادة الثالثة ينشر هذا القرار في الجريدة الرسمية، ويعمل به من اليوم التالي لتاريخ نشره.
Num Articles: 3
Word Count Articles: 42
avg_words_per_article: 13
articles: المادة الأولى : يعمل بأحكام النظام الأكاديمي لكلية العلوم الشرعية، المرفق.
----------------
المادة الثانية : يلغى كل ما يخالف هذا القرار، أو يتعارض مع أحكامه.
----------------
المادة الثالثة : ينشر هذا القرار في الجريدة الرسمية، ويعمل به من اليوم التالي لتاريخ نشره.
--------------------------------------------------------------------------------
نمونه 4447:
Text: نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على النظام الأساسي للدولة الصادر بالمرسوم السلطاني رقم ١٠١ / ٩٦، وعلى ال

In [31]:
df_clean[['num_articles', 'word_count_articles', 'avg_words_per_article']].describe()


,num_articles,word_count_articles,avg_words_per_article
count,7102.000000,7102.000000,7102.000000
mean,4.191777,182.127992,29.431287
std,14.195364,953.851661,45.805649
min,1.000000,14.000000,8.000000
25%,2.000000,37.000000,15.000000
50%,3.000000,53.000000,19.000000
75%,4.000000,91.000000,26.000000
max,585.000000,29777.000000,1133.000000


In [32]:
#must be remove from df_clean
df_clean = df_clean[(df_clean['avg_words_per_article'] <= 100) & (df_clean['num_articles'] <= 10)]


In [33]:
article_q3 = df_clean[(df_clean['word_count_articles'] <= 200)]

# for i, row in article_q3.iterrows():
#     print("\n-----------------")
#     print("link:", row['canonical_link'])
#     print("Title:", row['title'])
#     print("Word count:", row['word_count'])
#     print("Text :")
#     print(row['text'])

print("-----------")
print("total  :", len(article_q3))


-----------
total  : 6352


In [34]:
article_q3[['num_articles', 'word_count_articles', 'avg_words_per_article']].describe()


,num_articles,word_count_articles,avg_words_per_article
count,6352.000000,6352.000000,6352.000000
mean,2.798489,60.431518,20.361933
std,1.016791,34.695296,9.078941
min,1.000000,14.000000,8.000000
25%,2.000000,35.000000,15.000000
50%,3.000000,49.000000,18.000000
75%,3.000000,74.000000,23.000000
max,10.000000,200.000000,99.000000


In [35]:
for i, row in article_q3.head(10).iterrows():
    print("-----------------")
    # print("link:", row['canonical_link'])
    # print("date:", row['date'])
    # print("Title:", row['title'])
    # print("title_semantic:", row['title_semantic'])
    print("Text after removing footer :")
    print(row['text'])
    print("PREAMBLE:")
    print(row['PREAMBLE'])
    print("articles:")
    print(row['articles'])
    # print("-" * 80)
    # print("footer:")
    # print(row['footer'])


-----------------
Text after removing footer :
نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على عقد تأسيس مكتب عمان الوطني للاستثمارات الهندسية والتخطيط العمراني المؤرخ في ١٤ يناير ١٩٧٤. قررنا المادة ١ الموافقة على تأسيس مكتب للاستثمارات الهندسية باسم “مكتب عمان الوطني للاستثمارات الهندسية والتخطيط العمراني”، ورخصنا له بمزاولة أعماله في السلطنة. المادة ٢ تدعيما للمكتب في مزاولة أعماله تسند إليه الدولة الدراسات التخطيطية الهندسية والمعمارية اللازمة لمشاريع السلطنة والإشراف على تنفيذها وذلك: أ) إما بالأمر المباشر، ب) وإما عن طريق المناقصات المحلية ويكون شأنه في ذلك شأن المكاتب الوطنية، ج) وإما بمنحه الأفضلية في الأحوال التي يستدعي فيها الأمر مناقصات عالمية وذلك ما لم تتجاوز أسعاره عن ١٠% من قيمة العطاءات الأخرى المقدمة. المادة ٣ ينشر هذا المرسوم في الجريدة الرسمية ويعتبر نافذا من تاريخ التوقيع عليه.
PREAMBLE:
نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على عقد تأسيس مكتب عمان الوطني للاستثمارات الهندسية والتخطيط العمراني المؤرخ في ١٤ يناير ١٩٧٤
articles:
المادة ١ : الموافقة على تأسيس مكتب للاست

In [36]:
data_1 = df_clean[(df_clean['word_count_articles'] <= 200)].copy()
len(data_1)

6352

In [37]:
import re

def create_full_chunk(row):
    # 1. دریافت مقادیر و تبدیل به رشته (برای جلوگیری از خطای None)
    title = str(row.get('title', '')).strip()  #
    topic = str(row.get('title_semantic', '')).strip()
    preamble = str(row.get('PREAMBLE', '')).strip()
    raw_articles = str(row.get('articles', '')).strip()

    # 2. پاکسازی بخش مواد (Articles Cleaning)
    # این دستور تمام خطوطی که فقط شامل خط تیره هستند را با دو اینتر جایگزین می‌کند
    # پترن: \n (شروع خط جدید) + خط تیره‌های متوالی + \n (پایان خط)
    clean_articles = re.sub(r'\n\s*-+\s*\n', '\n\n', raw_articles)

    # 3. ساخت قالب نهایی
    chunk_text = (
        f"[TITLE]\n{title}\n\n"
        f"[TOPIC]\n{topic}\n\n"
        f"[PREAMBLE]\n{preamble}\n\n"
        f"[ARTICLES]\n{clean_articles}"
    )

    return chunk_text


data_1['chunks'] = data_1.apply(create_full_chunk, axis=1)
print(data_1['chunks'].iloc[10])

[TITLE]
مرسوم سلطاني رقم ٣٨ / ٧٤ بشأن استبدال جميع وحدات الوزن المستعملة في السلطنة بنظام الجرام والكيلوجرام والطن

[TOPIC]
مرسوم | استبدال | جميع وحدات الوزن المستعملة في السلطنة بنظام الجرام والكيلوجرام والطن

[PREAMBLE]
نحن قابوس بن سعيد سلطان عمان بناء على ما عرضه علينا وزير التنمية، وتحقيقا للمنفعة العامة، وتنظيمها لعمليات البيع والشراء وتمشيا مع القواعد العصرية للموازين

[ARTICLES]
المادة ١ : تستبدل جميع وحدات الوزن المستعملة حاليا في السلطنة بنظام الجرام والكيلوجرام والطن حيث يكون كل ألف جرام يساوي كيلوجرام واحد وكل ألف كيلوجرام يساوي طن واحد.

المادة ٢ : تصدر دائرة الضبط والجودة والمواصفات التابعة للمديرية العامة للصناعة بوزارة التنمية جميع قطع الوزن بالنظام الجديد المنصوص عليه أعلاه.

المادة ٣ : على جميع أصحاب الدكاكين والمحلات التجارية والمتعاملين بالوزن في كافة أنحاء السلطنة تطبيق استبدال الأوزان القديمة بواسطة البلديات المعنية أو مكاتب الولاة أو مركز الإرشاد الزراعي، في مدة أقصاها الخامس عشر من شهر نوفمبر (تشرين الثاني) سنة ١٩٧٤.

المادة ٤ : كل من يتعامل بعد هذا التاريخ بأي

In [38]:
data_1['word_count_chunks'] = data_1['chunks'].apply(
    lambda x: len(x.split()) if isinstance(x, str) else 0
)


In [39]:
data_1['word_count_chunks'].describe()


count    6352.000000
mean      140.841310
std        49.533754
min        61.000000
25%       106.000000
50%       132.000000
75%       161.000000
max       770.000000
Name: word_count_chunks, dtype: float64

In [40]:
data_1[(data_1['word_count_chunks'] >= 300)]['word_count_chunks'].describe()


count     65.000000
mean     362.184615
std       76.777864
min      300.000000
25%      320.000000
50%      335.000000
75%      376.000000
max      770.000000
Name: word_count_chunks, dtype: float64

In [41]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

data_1['token_count'] = data_1['chunks'].apply(
    lambda x: len(enc.encode(x)) if isinstance(x, str) else 0
)

data_1['token_count'].describe()

count    6352.000000
mean      589.526134
std       217.237071
min       225.000000
25%       439.000000
50%       552.000000
75%       678.000000
max      3155.000000
Name: token_count, dtype: float64

In [42]:
data_1.iloc[0]

json_link                        https://qanoon.om/wp-json/wp/v2/posts/110
short_link                                        https://qanoon.om/?p=110
canonical_link                         https://qanoon.om/p/1974/rd1974002/
date                                                   1974-01-14 00:00:00
title                    مرسوم سلطاني رقم ٢ / ٧٤ بالموافقة على تأسيس مك...
text                     نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على ع...
html                     <p><a class="pdf-link" href="http://data.qanoo...
error                                                                  NaN
text_word_count                                                        122
title_word_count                                                        16
title_semantic           مرسوم | تصديق | علي انشاء مكتب عمان الوطني للا...
footer                   صدر في: ٣٠ ذي الحجة ١٣٩٣هـ الموافق: ١٤ يناير ١...
articles                 المادة ١ : الموافقة على تأسيس مكتب للاستثمارات...
num_articles             

In [43]:
data_1[(data_1['token_count'] > 700)][['token_count','num_articles']].describe()


,token_count,num_articles
count,1424.000000,1424.000000
mean,902.419242,3.682584
std,213.490229,1.336689
min,701.000000,1.000000
25%,762.000000,3.000000
50%,841.000000,3.000000
75%,969.250000,4.000000
max,3155.000000,10.000000


In [44]:
data_2 = data_1[(data_1['token_count'] > 700)].copy()


In [45]:
print(data_2.iloc[10]['chunks'])

[TITLE]
مرسوم سلطاني رقم ٣١ / ٧٥ بإجراء تنقلات بين بعض السفراء العمانيين المعتمدين في الخارج

[TOPIC]
مرسوم | None |  تنقلات بين بعض السفراء العمانيين المعتمدين في الخارج

[PREAMBLE]
نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على المادتين (١٠)، (١٤) من قانون السلكين الدبلوماسي والقنصلي رقم ٥٠ / ٧٤، الصادر بتاريخ ٧ / ١٢ / ١٩٧٤، وبناء على ما عرضه علينا وزير الدولة للشؤون الخارجية

[ARTICLES]
المادة ١ : ينقل السفير السيد / شبيب بن تيمور آل سعيد من سفارتنا لدى جمهورية باكستان الإسلامية ويعين سفيرا فوق العادة ومفوضا لسلطنة عمان لدى المملكة المغربية.

المادة ٢ : ينقل السفير / سالم محمد الغيلاني من الديوان العام لوزارة الخارجية ويعين سفيرا فوق العادة مفوضا لسلطنة عمان لدى جمهورية باكستان الإسلامية.

المادة ٣ : ينقل السفير / ابراهيم بن حمد الحارثي من سفارتنا لدى جمهورية الجزائر الديمقراطية الشعبية ويعين سفيرا فوق العادة مفوضا لسلطنة عمان لدى المملكة العربية السعودية.

المادة ٤ : ينقل السفير / هلال بن علي الخليلي من سفارتنا لدى المملكة العربية السعودية إلى الديوان العام بوزارة الخارجية.

المادة ٥

In [46]:
import pandas as pd
import re

# فرض می‌کنیم تابعی برای شمارش دقیق توکن‌ها دارید، اگر نه از این تابع ساده استفاده کنید:
def count_tokens(text):
    # این را با متد اصلی خودتان (مثلاً tiktoken) جایگزین کنید
    return len(text.split())

def process_chunks(row):
    text = row['chunks']
    num_articles = row['num_articles']
    token_count = row['token_count']

    # ۱. استخراج بخش‌های ثابت (Header) و بخش بدنه (Articles)
    # با استفاده از Regex بخش‌ها را جدا می‌کنیم
    header_pattern = r"(\[TITLE\].*?\[TOPIC\].*?\[PREAMBLE\].*?)\n\s*\[ARTICLES\]"
    match = re.search(header_pattern, text, re.DOTALL)

    if not match:
        return [text] # اگر ساختار یافت نشد کل متن را برگردان

    header = match.group(1)
    articles_section = text.split("[ARTICLES]")[-1].strip()

    # جدا کردن ماده‌ها (با فرض اینکه با دو اینتر یا خط خالی جدا شده‌اند)
    articles_list = [a.strip() for a in articles_section.split('\n\n') if a.strip()]

    final_chunks = []

    # منطق شرطی شما:

    # --- حالت اول: num_articles >= 3 ---
    if num_articles >= 3:
        i = 0
        while i < len(articles_list):
            current_art = articles_list[i]
            # بررسی اینکه آیا می‌توانیم ماده بعدی را هم در این چانک جا دهیم (شرط زیر ۷۰۰ توکن)
            if i + 1 < len(articles_list):
                combined_two = current_art + "\n\n" + articles_list[i+1]
                full_test_text = f"{header}\n\n[ARTICLES]\n{combined_two}"

                if count_tokens(full_test_text) <= 700:
                    final_chunks.append(full_test_text)
                    i += 2 # دو ماده مصرف شد
                else:
                    final_chunks.append(f"{header}\n\n[ARTICLES]\n{current_art}")
                    i += 1 # یک ماده مصرف شد
            else:
                final_chunks.append(f"{header}\n\n[ARTICLES]\n{current_art}")
                i += 1

    # --- حالت دوم: num_articles == 2 ---
    elif num_articles == 2:
        for art in articles_list:
            final_chunks.append(f"{header}\n\n[ARTICLES]\n{art}")

    # --- حالت سوم: num_articles == 1 ---
    elif num_articles == 1:
        if token_count <= 900:
            final_chunks.append(text)
        else:
            # تقسیم داخلی ماده بر اساس پاراگراف (Paragraph-level)
            paragraphs = [p.strip() for p in articles_list[0].split('\n') if p.strip()]
            temp_para_group = ""
            for p in paragraphs:
                test_para = (temp_para_group + "\n" + p).strip()
                full_test_text = f"{header}\n\n[ARTICLES]\n{test_para}"

                # اینجا حد آستانه را مثلا ۶۰۰ توکن می‌گیریم که چانک‌ها خیلی بزرگ نشوند
                if count_tokens(full_test_text) <= 700:
                    temp_para_group = test_para
                else:
                    if temp_para_group:
                        final_chunks.append(f"{header}\n\n[ARTICLES]\n{temp_para_group}")
                    temp_para_group = p
            if temp_para_group:
                final_chunks.append(f"{header}\n\n[ARTICLES]\n{temp_para_group}")

    else:
        final_chunks.append(text)

    return final_chunks

# اعمال تابع روی دیتافریم
data_2['chunks'] = data_2.apply(process_chunks, axis=1)

# باز کردن لیست به ردیف‌های جداگانه (Explode)
data_2 = data_2.explode('chunks').reset_index(drop=True)

# به‌روزرسانی تعداد توکن‌های جدید برای هر چانک
data_2['token_count'] = data_2['chunks'].apply(count_tokens)

# نمایش نتیجه
print(data_2[['chunks', 'token_count']].head())

                                              chunks  token_count
0  [TITLE]\nمرسوم سلطاني رقم ٣٧ / ٧٤ بتعديل بعض م...          122
1  [TITLE]\nمرسوم سلطاني رقم ٣٧ / ٧٤ بتعديل بعض م...          118
2  [TITLE]\nمرسوم سلطاني رقم ٣٨ / ٧٤ بشأن استبدال...          112
3  [TITLE]\nمرسوم سلطاني رقم ٣٨ / ٧٤ بشأن استبدال...          141
4  [TITLE]\nمرسوم سلطاني رقم ٣٨ / ٧٤ بشأن استبدال...           92


In [47]:
# ۱. پردازش data_2 (طبق کد قبلی)
data_2['split_chunks'] = data_2.apply(process_chunks, axis=1)
data_final_long = data_2.explode('split_chunks').reset_index(drop=True)
data_final_long['new_token_count'] = data_final_long['split_chunks'].apply(count_tokens)

# ۲. آماده‌سازی بخش Long برای ادغام
# حذف ستون‌های قدیمی و جایگزینی با مقادیر جدید
data_final_long = data_final_long.drop(columns=['chunks', 'token_count'])
data_final_long = data_final_long.rename(columns={'split_chunks': 'chunks', 'new_token_count': 'token_count'})

# ۳. استخراج بخش Short از دیتافریم اصلی
df_short = data_1[data_1['token_count'] <= 700].copy()

# ۴. مرج نهایی
df_final = pd.concat([df_short, data_final_long], ignore_index=True)

# نمایش نتیجه نهایی
print(f"تعداد ردیف‌های اولیه: {len(data_1)}")
print(f"تعداد ردیف‌های نهایی: {len(df_final)}")

تعداد ردیف‌های اولیه: 6352
تعداد ردیف‌های نهایی: 8141


In [48]:
df_final.head()

,json_link,short_link,canonical_link,date,title,text,html,error,text_word_count,title_word_count,title_semantic,footer,articles,num_articles,word_count_articles,avg_words_per_article,PREAMBLE,chunks,word_count_chunks,token_count
0,https://qanoon.om/wp-json/wp/v2/posts/110,https://qanoon.om/?p=110,https://qanoon.om/p/1974/rd1974002/,1974-01-14,مرسوم سلطاني رقم ٢ / ٧٤ بالموافقة على تأسيس مك...,نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على ع...,"<p><a class=""pdf-link"" href=""http://data.qanoo...",NaN,122,16,مرسوم | تصديق | علي انشاء مكتب عمان الوطني للا...,صدر في: ٣٠ ذي الحجة ١٣٩٣هـ الموافق: ١٤ يناير ١...,المادة ١ : الموافقة على تأسيس مكتب للاستثمارات...,3,103,33,نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على ع...,[TITLE]\nمرسوم سلطاني رقم ٢ / ٧٤ بالموافقة على...,157,674
1,https://qanoon.om/wp-json/wp/v2/posts/111,https://qanoon.om/?p=111,https://qanoon.om/p/1974/rd1974003/,1974-01-14,مرسوم سلطاني رقم ٣ / ٧٤ بالموافقة على تأسيس “ش...,نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على ع...,"<p><a class=""pdf-link"" href=""http://data.qanoo...",NaN,118,13,مرسوم | تصديق | علي انشاء “شركة عمان الوطنية ل...,صدر في: ٢٠ ذي الحجة ١٣٩٣هـ الموافق: ١٤ يناير ١...,المادة ١ : الموافقة على تأسيس شركة للمقاولات ف...,3,97,31,نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على ع...,[TITLE]\nمرسوم سلطاني رقم ٣ / ٧٤ بالموافقة على...,144,599
2,https://qanoon.om/wp-json/wp/v2/posts/114,https://qanoon.om/?p=114,https://qanoon.om/p/1974/rd1974006/,1974-02-12,مرسوم سلطاني رقم ٦ / ٧٤ بخصوص إعادة تنظيم دائر...,نحن قابوس بن سعيد سلطان عمان نظرا لما تقتضيه ا...,"<p><a class=""pdf-link"" href=""http://data.qanoo...",NaN,68,12,مرسوم | تنظيم | اعادة تنظيم دائرة الحسابات وال...,صدر في: ١٩ محرم ١٣٩٤هـ الموافق: ١٢ فبراير ١٩٧٤...,المادة ١ : تسمى هذه الدائرة بدائرة المالية بدل...,4,53,12,نحن قابوس بن سعيد سلطان عمان نظرا لما تقتضيه ا...,[TITLE]\nمرسوم سلطاني رقم ٦ / ٧٤ بخصوص إعادة ت...,93,383
3,https://qanoon.om/wp-json/wp/v2/posts/116,https://qanoon.om/?p=116,https://qanoon.om/p/1974/rd1974011/,1974-03-12,مرسوم سلطاني رقم ١١ / ٧٤ بتعديل الفقرة (٣) من ...,بعد الاطلاع على قانون الجنسية العمانية رقم (١)...,"<p><a class=""pdf-link"" href=""http://data.qanoo...",NaN,94,20,مرسوم | تعديل | الفقرة من المادة من قانون الجن...,صدر في: ١٧ صفر ١٣٩٤هـ الموافق: ١٢ مارس ١٩٧٤م ح...,المادة ١ : تعدل\n----------------\nالمادة ٢ : ...,3,73,23,بعد الاطلاع على قانون الجنسية العمانية رقم (١)...,[TITLE]\nمرسوم سلطاني رقم ١١ / ٧٤ بتعديل الفقر...,132,494
4,https://qanoon.om/wp-json/wp/v2/posts/117,https://qanoon.om/?p=117,https://qanoon.om/p/1974/rd1974014/,1974-03-14,مرسوم سلطاني رقم ١٤ / ٧٤ بالتصديق على اتفاقية ...,نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على ا...,"<p><a class=""pdf-link"" href=""http://data.qanoo...",NaN,73,20,مرسوم | تصديق | علي اتفاقية انشاء المصرف الدول...,صدر في: ٢٠ صفر١٣٩٤هـ الموافق: ١٤ مارس ١٩٧٤م قا...,المادة ١ : التصديق على اتفاقية تأسيس المصرف ال...,2,34,16,نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على ا...,[TITLE]\nمرسوم سلطاني رقم ١٤ / ٧٤ بالتصديق على...,115,483


In [49]:
df_final['token_count'].describe()

count    8141.000000
mean      362.615281
std       190.773342
min        52.000000
25%       161.000000
50%       385.000000
75%       534.000000
max       770.000000
Name: token_count, dtype: float64

In [50]:
print(df_final.iloc[10]['chunks'])

[TITLE]
مرسوم سلطاني رقم ٤١ / ٧٤ بتشكيل مجلس للتنمية

[TOPIC]
مرسوم | تشكيل | مجلس للتنمية

[PREAMBLE]
نحن قابوس بن سعيد سلطان عمان نظرا لما تقتضيه المصلحة العامة ونظرا لما للتنمية من أهمية بالغة

[ARTICLES]
المادة ١ : تشكيل مجلس للتنمية برئاستنا وعضوية كل من: وزير الدولة للشؤون الخارجية مساعدا وزير الداخلية عضوا وزير الصحة عضوا وزير المواصلات عضوا وزير التجارة والصناعة عضوا وزير الزراعة والأسماك والنفط والمعادن عضوا مندوبا عن المالية عضوا

المادة ٢ : ينشر هذا المرسوم في الجريدة الرسمية ويعمل به من تاريخه.


In [51]:
print(df_final.iloc[0])

json_link                        https://qanoon.om/wp-json/wp/v2/posts/110
short_link                                        https://qanoon.om/?p=110
canonical_link                         https://qanoon.om/p/1974/rd1974002/
date                                                   1974-01-14 00:00:00
title                    مرسوم سلطاني رقم ٢ / ٧٤ بالموافقة على تأسيس مك...
text                     نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على ع...
html                     <p><a class="pdf-link" href="http://data.qanoo...
error                                                                  NaN
text_word_count                                                        122
title_word_count                                                        16
title_semantic           مرسوم | تصديق | علي انشاء مكتب عمان الوطني للا...
footer                   صدر في: ٣٠ ذي الحجة ١٣٩٣هـ الموافق: ١٤ يناير ١...
articles                 المادة ١ : الموافقة على تأسيس مكتب للاستثمارات...
num_articles             

In [52]:
df_final.columns.to_list()

['json_link',
 'short_link',
 'canonical_link',
 'date',
 'title',
 'text',
 'html',
 'error',
 'text_word_count',
 'title_word_count',
 'title_semantic',
 'footer',
 'articles',
 'num_articles',
 'word_count_articles',
 'avg_words_per_article',
 'PREAMBLE',
 'chunks',
 'word_count_chunks',
 'token_count']

In [53]:
df_final["year"] = df_final["date"].dt.year
df_final["id"] = (
    "rd"
    + df_final["year"].astype(str)
    + df_final.groupby("year").cumcount().add(1).astype(str).str.zfill(3)
)


In [54]:
n_samples = 200
random_samples = df_final.sample(n=n_samples)
for i, row in random_samples.head(300).iterrows():
    # print("link:", row['canonical_link'])
    # print("date:", row['date'])
    print("Title:", row['title'])
    # print("title_semantic:", row['title_semantic'])
    # print("Text after removing footer :")
    # print(row['text'])
    # print("PREAMBLE:")
    # print(row['PREAMBLE'])
    # print("articles:")
    # print(row['articles'])
    # print("-" * 80)
    # print("footer:")
    # print(row['footer'])


Title: مرسوم سلطاني رقم ١٠٩ / ٢٠٠٦ بالتصديق على اتفاقية التجارة الحرة بين حكومة سلطنة عمان وحكومة الولايات المتحدة الأمريكية
Title: وزارة القوى العاملة: قرار وزاري رقم ١ / ٢٠١٤ بتشكيل لجنة لبحث المطالب العمالية
Title: مرسوم سلطاني رقم ٧٤ / ٢٠٠٠ بتعديل الهيكل التنظيمي لوزارة التربية والتعليم
Title: وزارة الثروة الزراعية والسمكية وموارد المياه: قرار وزاري رقم ٢ / ٢٠٢٥ بتحديد الجزاءات الإدارية على المخالفين لقانون مزاولة المهن الطبية البيطرية واللوائح والقرارات الصادرة تنفيذا له
Title: مرسوم سلطاني رقم ٣٩ / ٨٥ بمنح امتيازات لبنك عمان للزراعة والأسماك
Title: وزارة الأوقاف والشؤون الدينية: قرار وزاري رقم ٥٥٧ / ٢٠٢٠ بإصدار اللائحة التنظيمية لشؤون الحج
Title: مرسوم سلطاني رقم ١ / ٢٠١٦ باعتماد خطة التنمية الخمسية التاسعة ٢٠١٦-٢٠٢٠م
Title: وزارة القوى العاملة: قرار وزاري رقم ٢٨٩ / ٢٠١٧ بإيقاف التصريح باستقدام القوى العاملة غير العمانية بصفة مؤقتة في نشاط مراكز تنمية المهارات الذهنية
Title: وزارة التجارة والصناعة: قرار وزاري رقم ٩٥ / ٩٦ بإلغاء القرار الوزاري رقم ١٢٨ / ٩٥
Title: الهيئة العامة للم

In [55]:
import pandas as pd
import re

# تابعی برای استخراج نوع و شماره قانون
def extract_law_info(text):
    if pd.isna(text):
        return pd.Series([None, None])

    # الگوی رجکس برای پیدا کردن شماره بعد از کلمه "رقم"
    # [\d٠-٩] هم اعداد انگلیسی و هم اعداد فارسی/عربی را پوشش می‌دهد
    match = re.search(r'رقم\s+([\d٠-٩]+\s*/\s*[\d٠-٩]+)', str(text))

    if match:
        law_number = match.group(1).strip() # استخراج گروه اعداد (مثلا: ٤١ / ٧٤)

        # پیدا کردن متن قبل از کلمه "رقم" برای پیدا کردن نوع قانون
        # text[:match.start()] تمام متن قبل از شروع پترن "رقم ..." را می‌دهد
        pre_text = text[:match.start()].strip()

        # مدیریت حالتی که نام وزارتخانه با دو نقطه جدا شده است
        # مثال: "وزارة الداخلية: قرار وزاري" -> فقط "قرار وزاري" را می‌خواهیم
        if ':' in pre_text:
            law_type = pre_text.split(':')[-1].strip()
        else:
            law_type = pre_text

        return pd.Series([law_type, law_number])
    else:
        # اگر الگوی "رقم ..." پیدا نشد
        return pd.Series([None, None])

# اعمال تابع روی ستون title و ساخت دو ستون جدید
df_final[['law_type', 'law_number']] = df_final['title'].apply(extract_law_info)

# نمایش نتیجه برای بررسی
print(df_final[['title', 'law_type', 'law_number']].head())

                                               title      law_type law_number
0  مرسوم سلطاني رقم ٢ / ٧٤ بالموافقة على تأسيس مك...  مرسوم سلطاني     ٢ / ٧٤
1  مرسوم سلطاني رقم ٣ / ٧٤ بالموافقة على تأسيس “ش...  مرسوم سلطاني     ٣ / ٧٤
2  مرسوم سلطاني رقم ٦ / ٧٤ بخصوص إعادة تنظيم دائر...  مرسوم سلطاني     ٦ / ٧٤
3  مرسوم سلطاني رقم ١١ / ٧٤ بتعديل الفقرة (٣) من ...  مرسوم سلطاني    ١١ / ٧٤
4  مرسوم سلطاني رقم ١٤ / ٧٤ بالتصديق على اتفاقية ...  مرسوم سلطاني    ١٤ / ٧٤


In [56]:
n_samples = 200
random_samples = df_final.sample(n=n_samples)
for i, row in random_samples.head(300).iterrows():
    print("Title:", row['title'])
    print("law_type:", row['law_type'])
    print("law_number:", row['law_number'])



Title: مرسوم سلطاني رقم ١١ / ٢٠١٣ بتعديل بعض أحكام نظام المجلس الأعلى للتخطيط
law_type: مرسوم سلطاني
law_number: ١١ / ٢٠١٣
Title: مرسوم سلطاني رقم ٤٤ / ٢٠٠٠ بالتصديق على اتفاقية التشجيع والحماية المتبادلة للاستثمارات بين حكومة سلطنة عمان وحكومة الجمهورية الجزائرية الديمقراطية الشعبية
law_type: مرسوم سلطاني
law_number: ٤٤ / ٢٠٠٠
Title: مرسوم سلطاني رقم ٥٥ / ٩٨ بالتصديق على اتفاقية تجنب الازدواج الضريبي على الدخل الناشئ من النقل الجوي الدولي وملحقها بين حكومة سلطنة عمان وحكومة جمهورية سنغافورة
law_type: مرسوم سلطاني
law_number: ٥٥ / ٩٨
Title: مرسوم سلطاني رقم ٦٢ / ٩٧ بإضافة مادة جديدة إلى المرسوم السلطاني رقم (٧٩ / ٨١)
law_type: مرسوم سلطاني
law_number: ٦٢ / ٩٧
Title: وزارة التنمية الاجتماعية: قرار وزاري رقم ١٢٥ / ٢٠١٩ بإصدار اللائحة التنفيذية لقانون الطفل
law_type: قرار وزاري
law_number: ١٢٥ / ٢٠١٩
Title: وزارة الإسكان: قرار وزاري رقم ٥٣ / ٢٠١٣ بإصدار اللائحة التنظيمية لضوابط تخطيط الأراضي
law_type: قرار وزاري
law_number: ٥٣ / ٢٠١٣
Title: مرسوم سلطاني رقم ١٧ / ٢٠١٩ بتعيين قاضيين في المح

In [57]:
n_samples = 20
random_samples = df_final.sample(n=n_samples)
for i, row in random_samples.head(300).iterrows():
    print("Title:", row['title'])
    print("title_semantic:", row['title_semantic'])
    print("PREAMBLE:", row['PREAMBLE'])
    print()

Title: مرسوم سلطاني رقم ٤٩ / ٨٣ بالتصديق على زيادة حصة السلطنة في حقوق السحب الخاص في صندوق النقد الدولي
title_semantic: مرسوم | تصديق | علي زيادة حصة السلطنة في حقوق تجريد الخاص في صندوق النقد الدولي
PREAMBLE: نحن قابوس بن سعيد سلطان عمان بعد الاطلاع على المرسوم السلطاني رقم ٢٦ / ٧٥ بإصدار قانون تنظيم الجهاز الإداري للدولة وتعديلاته، وعلى القانون المصرفي العماني رقم ٧ / ٧٤، وعلى المرسوم السلطاني رقم ٣١ / ٧١ بالتفويض بطلب انضمام سلطنة عمان إلى عضوية بعض المنظمات الدولية، وعلى المرسوم السلطاني رقم ٣٤ / ٧٧ بشأن التصديق على موافقة مجلس محافظي البنك المركزي العماني على التعديلات التي طرأت على مواد اتفاقية صندوق النقد الدولي، وعلى المرسوم السلطاني رقم ٥٠ / ٧٨، والمرسوم السلطاني رقم ١ / ٨١ بالتصديق على زيادة حصة السلطنة في حقوق السحب الخاص في صندوق النقد الدولي، وعلى قرار مجلس محافظي صندوق النقد الدولي رقم ٣٤ / ٢ الصادر في ١١ / ١٢ / ١٩٧٨، وبناء على ما تقتضيه المصلحة العامة

Title: وزارة العدل والشؤون القانونية: استدراك
title_semantic: None
PREAMBLE: تنوه وزارة العدل والشؤون القانونية إلى أنه

In [58]:
import pandas as pd
import re

# 1. تابع نرمال‌سازی برای یکدست کردن متن
def normalize_arabic(text):
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)
    return text

# 2. لیست توسعه‌یافته کلمات توقف (بر اساس خروجی‌های شما)
custom_stopwords = set([
    # کلمات پایه قبلی (با نرمال‌سازی)
    'مرسوم', 'سلطاني', 'رقم', 'قرار', 'وزاري', 'وزاره', 'قانون',
    'نحن', 'قابوس', 'بن', 'سعيد', 'سلطان', 'عمان', 'بعد', 'الاطلاع',
    'علي', 'في', 'من', 'الي', 'عن', 'بناء', 'ما', 'تقتضيه', 'المصلحة',
    'العامة', 'الصادر', 'بالمرسوم', 'النظام', 'الاساسي', 'للدوله',
    'وتعديلاته', 'استنادا', 'اللايحه', 'التنفيذيه', 'القرار', 'وتحديد',
    'اختصاصاتها', 'واعتماد', 'هيكلها', 'التنظيمي', 'تقرر', 'بشان',
    'تاريخ', 'الموافق', 'م', 'ه', 'سنه', 'لعام', 'لايحه', 'اصدار',
    'باصدار', 'بتعديل', 'تعديل', 'وبناء', 'والي', 'وعلي', 'التصديق', 'بالتصديق',

    # --- موارد جدید اضافه شده بر اساس نمونه‌های شما ---
    # نام‌های جدید
    'هيثم', 'طارق', 'ال',

    # افعال و اسامی اجرایی (Procedural)
    'تعيين', 'بتعيين', 'نقل', 'بنقل', 'ترقيه', 'بترقيه', 'انشاء', 'بانشاء',
    'الغاء', 'بالغاء', 'اجازه', 'باجازه', 'تجديد', 'بتجديد', 'تمديد', 'بتمديد',
    'تطبيق', 'بتطبيق', 'تنظيم', 'بتنظيم', 'تقرير', 'بتقرير', 'اعاده', 'باعاده',
    'تشكيل', 'بتشكيل', 'منح', 'بمنح', 'فرض', 'بفرض', 'حظر', 'بحظر',
    'استحداث', 'باستحداث', 'اعتبار', 'باعتبار', 'تخويل', 'بتخويل',
    'تسميه', 'بتسميه', 'اضافه', 'باضافه', 'استبدال', 'باستبدال',
    'تصريح', 'باستقدام', 'ايقاف', 'بايقاف', 'توقيع', 'بالتوقيع', 'تفويض', 'بالتفويض',

    # کلمات عمومی حقوقی و اداری (Generic Entities)
    'حكومه', 'سلطنه', 'جمهوريه', 'دوله', 'مجلس', 'لجنه', 'هيي', 'موسسه', 'شركه', 'وشركه',
    'مكتب', 'دايره', 'مديريه', 'قسم', 'فرع', 'منطقه', 'محافظه', 'ولايه',
    'ماده', 'الماده', 'بند', 'فقره', 'احكام', 'الاحكام', 'نصوص',
    'اتفاقيه', 'مذكره', 'تفاهم', 'بروتوكول', 'وثيقه', 'عقد',
    'رئيس', 'نايب', 'وكيل', 'مدير', 'عام', 'امين', 'عضو', 'اعضاء', 'سفير',
    'الدرجه', 'الخاصه', 'الماليه', 'الاداري', 'التنفيذي', 'المرفق', 'الملحق',
    'بعض', 'كل', 'جميع', 'غير', 'شوي', 'لدى', 'بين', 'حول', 'ضمن', 'ذات',
    'الجهاز', 'السلكين', 'الدبلوماسي', 'والقنصلي', 'الموحد', 'الرسميه', 'الجريده',
    'الاول', 'الثاني', 'الجزء', 'المتبقي', 'مشروع', 'لمشروع', 'سند', 'صنه'
])

def extract_tags_optimized(row):
    # ترکیب ستون‌ها
    text_source = f"{str(row['title'])} {str(row['title_semantic'])} {str(row['PREAMBLE'])}"

    # پاکسازی اولیه
    text_clean = re.sub(r'[^\w\s]', ' ', text_source)
    text_clean = re.sub(r'\d+', '', text_clean)

    # نرمال‌سازی متن (بسیار مهم برای حذف دایره لغات تکراری)
    text_normalized = normalize_arabic(text_clean)

    words = text_normalized.split()

    tags = []
    seen = set()

    for word in words:
        original_word = word # کلمه اصلی نرمال شده
        processed_word = word

        # حذف پیشوندهای عربی (ال، ب، و، ل، ف) اگر طول کلمه اجازه دهد
        # مرحله 1: حذف "ال"
        if processed_word.startswith('ال') and len(processed_word) > 4:
            processed_word = processed_word[2:]

        # مرحله 2: حذف حروف ربط چسبیده (و، ف، ب، ل)
        # فقط در صورتی حذف می‌کنیم که کلمه باقی‌مانده هنوز معنی‌دار باشد (طول > 3)
        if processed_word.startswith(('و', 'ف', 'ب', 'ل')) and len(processed_word) > 4:
            processed_word = processed_word[1:]
            # دوباره چک کردن "ال" بعد از حذف حرف ربط (مثلاً "والتخطيط" -> "تخطيط")
            if processed_word.startswith('ال') and len(processed_word) > 4:
                processed_word = processed_word[2:]

        # چک کردن کلمه نهایی با لیست توقف
        if (processed_word not in custom_stopwords and
            original_word not in custom_stopwords and
            len(processed_word) > 3): # کلمات خیلی کوتاه (2 و 3 حرفي) معمولا نویز هستند

            if processed_word not in seen:
                tags.append(processed_word) # کلمه پردازش شده (بدون ال و...) را ذخیره می‌کنیم
                seen.add(processed_word)

    if not tags:
        return None

    # برگرداندن 6 تگ اول (تعداد کمتر اما با کیفیت‌تر)
    return tags[:6]

# اعمال روی دیتا فریم
df_final['semantic_tags'] = df_final.apply(extract_tags_optimized, axis=1)

# نمایش چند نمونه برای تست
print(df_final[['title', 'semantic_tags']].head(10))

                                               title  \
0  مرسوم سلطاني رقم ٢ / ٧٤ بالموافقة على تأسيس مك...   
1  مرسوم سلطاني رقم ٣ / ٧٤ بالموافقة على تأسيس “ش...   
2  مرسوم سلطاني رقم ٦ / ٧٤ بخصوص إعادة تنظيم دائر...   
3  مرسوم سلطاني رقم ١١ / ٧٤ بتعديل الفقرة (٣) من ...   
4  مرسوم سلطاني رقم ١٤ / ٧٤ بالتصديق على اتفاقية ...   
5  مرسوم سلطاني رقم ١٧ / ٧٤ بالتصديق على اتفاقية ...   
6  مرسوم سلطاني رقم ٣٠ / ٧٤ بتعيين سكرتير أول في ...   
7  مرسوم سلطاني رقم ٣١ / ٧٤ بتعيين مستشار في وزار...   
8  مرسوم سلطاني رقم ٣٢ / ٧٤ بتعيين وزير مفوض في و...   
9  مرسوم سلطاني رقم ٣٩ / ٧٤ بتعيين مستشار بوزارة ...   

                                     semantic_tags  
0  [موافقه, تاسيس, وطني, لاستشارات, هندسيه, تخطيط]  
1  [موافقه, تاسيس, طنيه, لمقاولات, تصديق, مقاولات]  
2         [خصوص, حسابات, خزينه, نظرا, مصلحه, عامه]  
3        [جنسيه, عماني, لسنه, عمانيه, عرضه, معالي]  
4        [تاسيس, مصرف, دولي, عربي, لتجاره, خارجيه]  
5   [تاسيس, عربي, لتنميه, اقتصاديه, افريقيا, نظام]  
6      [سكرت

In [59]:
n_samples = 200
random_samples = df_final.sample(n=n_samples)
for i, row in random_samples.head(300).iterrows():

    print("semantic_tags:", row['semantic_tags'])


semantic_tags: ['جنسيه', 'عمانيه', 'مصلحه', 'عامه']
semantic_tags: ['معتمد', 'مملكه', 'مغربيه', 'سفيرا', 'مقيم', 'اسلاميه']
semantic_tags: ['رييس', 'اداره', 'غرفه', 'تجاره', 'صناعه', 'مصلحه']
semantic_tags: ['تشجيع', 'حمايه', 'متبادله', 'لاستثمارات', 'مملكه', 'سويد']
semantic_tags: ['منفعه', 'عامه', 'تطوير', 'طينه', 'بخور', 'ملكيه']
semantic_tags: ['منظمه', 'لاستثمار', 'خليجي', 'شركات', 'تجاريه', 'ضريبه']
semantic_tags: ['داخليه', 'تحديد', 'رسوم', 'اثمان', 'التي', 'تحصلها']
semantic_tags: ['مساعد', 'زراء', 'مصلحه', 'عامه']
semantic_tags: ['وزير', 'محافظ', 'مسقط', 'ممارسه', 'نشاط', 'تداول']
semantic_tags: ['موازنه', 'عامه', 'تصديق', 'مالي', 'مصلحه']
semantic_tags: ['تحديد', 'رسوم', 'دعاوي', 'مدنيه', 'احوال', 'شخصيه']
semantic_tags: ['تجاره', 'صناعه', 'ترويج', 'استثمار', 'مبادي', 'حوكمه']
semantic_tags: ['امتياز', 'استكشاف', 'تعدين', 'عالميه', 'متكامله', 'لهندسه']
semantic_tags: ['دكتور', 'سالم', 'حمدان', 'سويد', 'اخزمي', 'كيلا']
semantic_tags: ['هييه', 'عامه', 'لكهرباء', 'مياه', 'اداري'

In [64]:
n_samples = 10
random_samples = df_final.sample(n=n_samples)
for i, row in random_samples.head(10).iterrows():
    print("chunks:", row['chunks'])
    print("="*60)

chunks: [TITLE]
هيئة تنظيم الاتصالات: قرار رقم ١ / ٢٠١٥ بتعديل بعض أحكام القرار رقم ١٣٣ / ٢٠٠٨ بإصدار لائحة تنظيم تسجيل واستخدام الترددات والأجهزة الراديوية وتحديد أسعارها

[TOPIC]
قرار | تعديل | بعض احكام القرار اصدار لائحة تنظيم تسجيل واستخدام الترددات والاجهزة الراديوية تحديد اسعارها

[PREAMBLE]
استنادا إلى قانون تنظيم الاتصالات الصادر بالمرسوم السلطاني رقم ٣٠ / ٢٠٠٢، وإلى اللائحة التنفيذية لقانون تنظيم الاتصالات الصادرة بالقرار رقم ١٤٤ / ٢٠٠٨، وإلى القرار رقم ١٣٣ / ٢٠٠٨ بإصدار لائحة تنظيم تسجيل واستخدام الترددات والأجهزة الراديوية وتحديد أسعارها، وإلى موافقة مجلس الإدارة في اجتماعه رقم ٣ / ٢٠١٤ بتاريخ ٢١ من يوليو ٢٠١٤م، وبناء على ما تقتضيه المصلحة العامة. تقرر

[ARTICLES]
المادة الثانية : يلغى كل ما يخالف هذا القرار، أو يتعارض مع أحكامه.

المادة الثالثة : ينشر هذا القرار في الجريدة الرسمية، ويعمل به من اليوم التالي لتاريخ نشره.
chunks: [TITLE]
مرسوم سلطاني رقم ٦٣ / ٩٥ بالتفويض في التوقيع على اتفاقية بين حكومة سلطنة عمان وحكومة جمهورية أوغندا بشأن الخدمات الجوية بين إقليميهما وما ور

In [61]:
# import json
# import pandas as pd
# import numpy as np
# # لیست نهایی برای ذخیره آبجکت‌ها
# json_output_list = []
#
# # پیمایش روی تک تک سطرهای دیتا فریم
# for index, row in df_final.iterrows():
#
#     # ---------------------------------------------------------
#     # 1. تولید ID طبق پترن درخواستی: rd1974002_chunk_01
#     # فرمول: rd + سال (4 رقم) + آی‌دی (3 رقم با پدینگ صفر) + _chunk_01
#     # ---------------------------------------------------------
#     try:
#         # سال
#         year_str = str(int(row["year"])) if pd.notna(row["year"]) else "0000"
#
#         # آی‌دی عددی داخل سال
#         if pd.notna(row["id"]):
#             id_val = int(row["id"]) if isinstance(row["id"], (int, float)) else int(str(row["id"]))
#             id_str = str(id_val).zfill(3)
#         else:
#             id_str = "000"
#
#         # شماره چانک بر اساس index دیتافریم (از 1 شروع می‌کنیم)
#         chunk_num = int(index) + 1
#         chunk_str = str(chunk_num).zfill(2)  # اگر دیتاست بزرگ است → 3
#
#         generated_id = f"rd{year_str}{id_str}_chunk_{chunk_str}"
#
#     except Exception:
#         generated_id = f"rd_error_{index}_chunk_{str(index + 1).zfill(2)}"
#
#
#     # ---------------------------------------------------------
#     # 2. ساخت دیکشنری برای هر سطر
#     # ---------------------------------------------------------
#     # مدیریت مقادیر NaN (چون JSON مقدار NaN را قبول نمی‌کند و باید null شود)
#     # تابع کمکی برای هندل کردن NaN
#
#
#     def get_val(val):
#         if val is None:
#             return None
#
#         # pandas Timestamp / datetime
#         if isinstance(val, (pd.Timestamp,)):
#             return val.isoformat()
#
#         # numpy datetime64
#         if isinstance(val, np.datetime64):
#             return pd.to_datetime(val).isoformat()
#
#         # list / tuple
#         if isinstance(val, (list, tuple)):
#             return val if len(val) > 0 else None
#
#         # numpy array
#         if hasattr(val, "tolist"):
#             return val.tolist()
#
#         # scalar NaN
#         if pd.isna(val):
#             return None
#
#         return val
#
#
#     record = {
#         "id": generated_id,
#         "text": get_val(row['text']),
#         "chunk": get_val(row['chunks']),
#         "metadata": {
#             "doc_id": get_val(row['id']),
#             "law_type": get_val(row['law_type']),
#             "law_number": get_val(row['law_number']),
#             "year": get_val(row['year']),
#             "date": get_val(row['date']),
#             "footer": get_val(row['footer']),
#             "title": get_val(row['title']),
#             "semantic_tags": get_val(row['semantic_tags']),
#             "canonical_link": get_val(row['canonical_link'])
#         }
#     }
#
#     json_output_list.append(record)
#
# # ---------------------------------------------------------
# # 3. ذخیره در فایل JSON
# # ---------------------------------------------------------
# output_filename = 'oman_laws_final_dataset.json'
#
# with open(output_filename, 'w', encoding='utf-8') as f:
#     # ensure_ascii=False باعث می‌شود متن فارسی/عربی درست ذخیره شود (نه به صورت یونیکد)
#     # indent=4 برای مرتب‌سازی و خوانایی فایل
#     json.dump(json_output_list, f, ensure_ascii=False, indent=4)
#
# print(f"فایل جیسون با موفقیت ایجاد شد: {output_filename}")
# print(f"تعداد رکوردها: {len(json_output_list)}")